# Artifact Evaluation — Improving Progressive Compression with Adaptive Interpolation and Coefficient Decomposition

This notebook reproduces the figures and tables in the paper. Run the cells top to bottom: setup and data preparation first, then run the experiments, then plot each figure/table from the produced results.

---

## TL;DR for reviewers

- **Core results (Fig. 6–10, Tables 3–4, Fig. 12)** run on a **single multi-core node** — no cluster required. We validated this on the Chameleon platform, on a *CHI@UC compute_gigaio* node with two AMD EPYC 7763 processors (2 sockets × 64 cores = 128 physical cores, 256 hardware threads). Using GNU `parallel` with 20 concurrent jobs, all core experiments finish in **about 2 hours** — well within the 8-hour review budget.
- **Fig. 11 is OPTIONAL.** It is a 1024-core cross-node MPI weak-scaling run and is *not* required to validate the paper's main claims (see the Fig. 11 note).
  Skip it unless you have a partition providing 1024 cores of a single architecture.
- Each experiment can be launched either through a **SLURM cluster** (batch submission) **or** on a **single node** (GNU `parallel`) — pick whichever matches your resources.

---

## Two ways to run the experiments

The core experiments are many small, independent per-field jobs. You can run them in either of two environments:

- **(A) Single node with GNU `parallel`** *(recommended; e.g. Chameleon)*. No scheduler needed. A helper runs all per-field jobs at a configurable concurrency (`MAX_PARALLEL_JOBS`, default 20). On our 128-core / 256-thread node, concurrency 20 was chosen to stay within memory rather than cores; adjust it to fit your node's memory. Measured: ~2 hours (≈100 min for the ablation jobs) for all core experiments.
- **(B) SLURM cluster**. Batch-submit one job per field via the provided `./ae_scripts/*.sh` and the notebook's submission cells. Each job requests `--ntasks=16`; with enough nodes the whole set runs concurrently.

Both paths produce the same result files; the plotting cells are identical.

> Run this notebook from the **ProDM** directory (the scripts assume that working directory).

---

## What you need to configure

Only a few things are cluster/environment specific; all are marked in the notebook:

1. **Storage path (`DATA_ROOT`)** — in *Data preparation*. Needs enough free space (see *Estimated time & storage*).

2. **Execution path** — choose (A) single-node `parallel` or (B) SLURM:
   - **(A)** set `MAX_PARALLEL_JOBS` (concurrent jobs, default 20) to fit your node's memory and cores.
   - **(B)** fill in `SBATCH_SETTINGS` (at least `--partition` and `-A`; add `--constraint`, `--gres`, `--qos`, etc. as your cluster requires).

3. **Node architecture / compilation** — the binaries must run on the node(s) you use. On homogeneous hardware, ignore this. On **heterogeneous** hardware (nodes with different CPU architectures), build on one node type and pin every job to that same type (via `--constraint`/`--nodelist` on SLURM, or by simply running on that node under path A); otherwise binaries may crash or give non-comparable timings.

4. **(Optional, Fig. 11 only)** `JHTDB_SBATCH_SETTINGS` + `MODULE_SETUP` — a separate configuration for the 1024-core MPI job, plus your cluster's MPI module name (e.g. `openmpi/4.1.5`). Only needed if you attempt the optional Fig. 11.

---

## Table of contents

### Part 1 - Core Reproduction (single node, required)
**Setup**
- Dependencies (run first)
- Imports & shared configuration

**Data preparation**
- Download & rearrange the four datasets (set `DATA_ROOT` here)
- Execution configuration — single-node `parallel` **or** `SBATCH_SETTINGS`

**Fig. 6 — S3D decompose / encoding time**

**Fig. 7–10 & Tables III–IV — ablation study (CESM, Miranda, SCALE, S3D)**
- Run ablation jobs (all datasets) — path A or B
- Fig. 7 — S3D, EB mode
- Fig. 8 — S3D, PSNR mode
- Fig. 9 — error bound vs bitrate (+ zoom)
- Fig. 10 — PSNR vs bitrate (+ zoom)
- Table III — average refactor time
- Table IV — average reconstruction time & bit-rate

**Fig. 12 — CESM Temperature visual comparison**
- Run `visualize_CESM_Temp.sh` (after the CESM ablation job)

### Part 2 - Optional: JHTDB Weak-Scaling Study (Fig.11)
**Fig. 11 — JHTDB weak scaling (OPTIONAL, Part 2)**
- All Part-2-specific settings (`JHTDB_ROOT`, `TOKEN`, `MODULE_SETUP`, `JHTDB_SBATCH_SETTINGS`) are set together in one cell right after this section's heading — nothing else in Part 2 requires further configuration.
- ⚠️ Optional: needs 1024 cores of a single architecture; not required for the paper's main claims.
- ⚠️ JHTDB's own data access is currently down (server-side); we provide the data via Globus instead.

---

## Estimated time & storage

- **Core experiments (Fig. 6–10, Tables 3–4, Fig. 12):** ~2 hours on a single 128-core / 256-thread node (GNU `parallel`, 20 concurrent jobs; ≈100 min for the ablation jobs); faster on a multi-node SLURM cluster. Comfortably within
  the 8-hour review budget.
- **Storage (core):** ~200 GB including refactored representations for the four small datasets.
- **Fig. 11 (optional):** ~2.3 TB (original JHTDB cubes + refactored representations + retrieved data) and 1024 cores; see the Fig. 11 note.

## Part 1 - Core Reproduction (single node, required)

### Setup — Install dependencies

Make sure all required packages are installed before proceeding.

In [ ]:
# =============================================================================
# Dependencies -- run this first.
# Checks that all required third-party packages are importable. If any are
# missing, it lists them and asks before installing (nothing is installed
# automatically). Standard-library modules used by the download step are
# imported at the end (no installation needed).
# =============================================================================
import importlib.util, sys

REQUIRED = {
    "numpy": "numpy",
    "matplotlib": "matplotlib",
    "scipy": "scipy",
    "requests": "requests",
}  # import_name -> pip_name

missing = [pip for mod, pip in REQUIRED.items() if importlib.util.find_spec(mod) is None]

if missing:
    print("Missing required packages:", ", ".join(missing))
    ans = input("Install them now with pip? [y/N] ").strip().lower()
    if ans == "y":
        import subprocess
        subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
        print("Done. Re-run this cell to confirm.")
    else:
        raise SystemExit("Please install the missing packages before continuing: "
                         + ", ".join(missing))
else:
    print("All required packages are available.")

import os
import re
import math
import tarfile
import urllib.request
import glob
import subprocess
import base64
import time
import random
import xml.etree.ElementTree as ET
from collections import defaultdict
from multiprocessing import Pool, Lock, Manager
import numpy as np
import shutil

In [ ]:
# ---- REVIEWER: define your storage path here --------------------------------
DATA_ROOT = "/path/to/your/storage"   # e.g. "/pscratch/<proj>/<user>/test_datasets"
# -----------------------------------------------------------------------------

AE_SCRIPTS = "./ae_scripts"        # where the template .sh files live

In [ ]:
# ---- REVIEWER: set your execution mode here --------------------------------
EXECUTION_MODE = "parallel"   # "slurm" or "parallel"

# --- settings only used when EXECUTION_MODE == "parallel" -------------------
JOBLIST_PATH = os.path.join(os.getcwd(), "joblist.txt")
JOBLOG_PATH = os.path.join(os.getcwd(), "run.log")
MAX_PARALLEL_JOBS = 20   # concurrent jobs
# -----------------------------------------------------------------------------

In [ ]:
# =============================================================================
# REVIEWER: cluster job settings (only needed if EXECUTION_MODE == "slurm")
# =============================================================================
#
# ⏭️  If you set EXECUTION_MODE = "parallel" above, you can SKIP this entire
#     cell — SBATCH_SETTINGS is not used in parallel mode. Just run this cell
#     as-is (with the defaults below) or skip running it altogether.
#
# ⚠️ IMPORTANT — architecture matching on heterogeneous clusters:
#   The precompiled/locally-built binaries may use architecture-specific
#   instructions (e.g. AVX-512). On a heterogeneous cluster (nodes with
#   different CPU architectures), a binary built on one node type may crash
#   ("Illegal instruction") or give non-comparable results on another.
#
#   Rule of thumb: BUILD and RUN on the SAME node architecture.
#     1. Pick one node type on your cluster and build the code there
#        (e.g. `salloc --constraint=<arch> ...`, then compile).
#     2. Set the SAME constraint below so every job is pinned to that node type.
#
#   On a homogeneous cluster (all nodes identical, e.g. MCC), you can leave
#   --constraint empty.
#
# HOW THIS WORKS:
#   SBATCH_SETTINGS may contain ANY #SBATCH option, not just those present in a
#   template. Options already in the template are filled in place; options that
#   are not (e.g. --constraint, --exclusive, --gres) are appended to the SBATCH
#   header. Leave an option as "" to skip it. For a valueless flag (e.g.
#   --exclusive), use a single space " " so it is emitted as a bare directive.
# -----------------------------------------------------------------------------


SBATCH_SETTINGS = {
    # --- required ---
    "--partition":  "",      # partition/queue, e.g. "normal" or "gpu"
    "-A":           "",      # your allocation account

    # --- notifications (leave --mail-user "" to drop the mail lines) ---
    "--mail-user":  "",      # your email

    # --- node architecture / placement (heterogeneous clusters) ---
    # Set --constraint to the SAME node type you compiled on. Leave "" on a
    # homogeneous cluster.
    "--constraint": "",      # e.g. "skylake", "cascadelake", "intel&avx512"

    # --- optional: uncomment and set as needed for your cluster ---
    # "--exclusive":       " ",        # whole-node allocation (valueless flag)
    # "--gres":            "",         # generic resources, e.g. "gpu:1"
    # "--mem":             "",         # memory per node, e.g. "64G"
    # "--mem-per-cpu":     "",         # memory per cpu, e.g. "12G"
    # "--nodelist":        "",         # force specific node(s)
    # "--exclude":         "",         # exclude specific node(s)
    # "--qos":             "",         # quality of service

    # --- optional: override resource defaults baked into the templates ---
    # "--time":   "03:00:00",
    # "--nodes":  "1",
    # "--ntasks": "16",
}
# -----------------------------------------------------------------------------

### Setup — Download and prepare datasets

Download the CESM, Miranda, SCALE, and S3D datasets and rearrange them. Takes around 22 minutes.

Please provide a path to a location with enough space to store these datasets.
We recommend using temporary/scratch storage on your cluster.

**Storage requirements:**
- Raw datasets: CESM (41.42 GB), Miranda (1.97 GB), SCALE (11.57 GB), S3D (8.38 GB) — about 63 GB in total.
- Including the refactored representations of these four datasets, the total requirement is roughly 200 GB.
- For Fig. 11, the JHTDB dataset alone is 512 GB, and additional space is needed to store its refactored representation. We therefore recommend a location with more than 2.3 TB of available storage if you'd like to run experiment for Fig. 11. The downloading and processing of the JHTDB dataset is placed at the Fig. 11 section.

In [ ]:
# =============================================================================
# Download the four small datasets (CESM, Miranda, SCALE, S3D) and rearrange
# them into a uniform per-dataset layout:
#
#   {DATA_ROOT}/{dataset}/data/{VAR}.dat              (raw field, float64)
#   {DATA_ROOT}/{dataset}/refactor/{VAR}_refactored/  (empty; filled later)
#
# Source fields are stored as float32 (CESM, SCALE) or float64 (Miranda, S3D);
# all are converted/written as double (.dat). Every variable present in each
# source dataset is converted. Requires os/sys/tarfile/urllib/shutil (imported
# in the dependency cell) and numpy.
#
# To save disk space, the downloaded .tar.gz archive and the extracted
# source-format directory are deleted once a dataset has been rearranged into
# data/ + refactor/. Set KEEP_SOURCE_ARCHIVES = True below if you'd rather
# keep them around (e.g. to re-run the rearrange step without re-downloading).
#
# >>> REVIEWER: set DATA_ROOT to a directory with enough free space. 
#     Raw data ~63 GB; with refactored representations ~183 GB total.
#     (With KEEP_SOURCE_ARCHIVES = False, peak usage is lower -- each
#     dataset's archive/source dir is cleaned up before the next one downloads.)
#     (JHTDB, used in Fig. 11, is 512 GB and must be obtained separately.)
# =============================================================================
if DATA_ROOT == "/path/to/your/storage":
    raise SystemExit("Please set DATA_ROOT to a valid storage path before running.")

KEEP_SOURCE_ARCHIVES = False   # True to keep the .tar.gz + extracted source dirs

# -----------------------------------------------------------------------------
# 1. Download + extract
# -----------------------------------------------------------------------------
BASE = "https://g-8d6b0.fd635.8443.data.globus.org/ds131.2/Data-Reduction-Repo/raw-data"

DOWNLOADS = {
    "CESM":    f"{BASE}/CESM-ATM/SDRBENCH-CESM-ATM-26x1800x3600.tar.gz",
    "Miranda": f"{BASE}/Miranda/SDRBENCH-Miranda-256x384x384.tar.gz",
    "SCALE":   f"{BASE}/SCALE_LETKF/SDRBENCH-SCALE-98x1200x1200.tar.gz",
    "S3D":     f"{BASE}/S3D/SDRBENCH-S3D.tar.gz",
}


def _progress(count, block_size, total_size):
    if total_size <= 0:
        return
    pct = min(count * block_size / total_size * 100, 100)
    sys.stdout.write(f"\r    {pct:5.1f}%")
    sys.stdout.flush()


def download_and_extract(name, url, dest_root):
    out_dir = os.path.join(dest_root, name)
    os.makedirs(out_dir, exist_ok=True)
    archive = os.path.join(out_dir, os.path.basename(url))

    if os.path.exists(archive):
        print(f"[{name}] archive already present, skipping download: {archive}")
    else:
        print(f"[{name}] downloading -> {archive}")
        urllib.request.urlretrieve(url, archive, reporthook=_progress)
        print()  # newline after progress bar

    print(f"[{name}] extracting into {out_dir}")
    with tarfile.open(archive, "r:gz") as tar:
        tar.extractall(path=out_dir)
    print(f"[{name}] download/extract done.\n")
    return archive


os.makedirs(DATA_ROOT, exist_ok=True)
ARCHIVE_PATHS = {}
for name, url in DOWNLOADS.items():
    ARCHIVE_PATHS[name] = download_and_extract(name, url, DATA_ROOT)

# -----------------------------------------------------------------------------
# 2. Rearrange into {dataset}/data + {dataset}/refactor
# -----------------------------------------------------------------------------
# Standard datasets: extracted sub-dir, field-file glob, source dtype, and how
# to derive the variable name from a filename (delimiter differs per dataset).
REARRANGE = {
    "CESM": {
        "subdir": "SDRBENCH-CESM-ATM-26x1800x3600",
        "pattern": "*.f32",
        "src_dtype": np.float32,
        # "CLDICE_1_26_1800_3600.f32" -> split on "_"
        "varname": lambda fn: fn.split("_")[0],
    },
    "Miranda": {
        "subdir": "SDRBENCH-Miranda-256x384x384",
        "pattern": "*.d64",
        "src_dtype": np.float64,
        # "density.d64" -> split on "."
        "varname": lambda fn: fn.split(".")[0],
    },
    "SCALE": {
        "subdir": "SDRBENCH-SCALE_98x1200x1200",
        "pattern": "*.f32",
        "src_dtype": np.float32,
        # "PRES-98x1200x1200.f32" -> split on "-"
        "varname": lambda fn: fn.split("-")[0],
    },
}

# S3D: one multi-variable file, sliced along axis 0. Variable order and dtype
# per SDRBENCH-S3D/template.txt (11x500x500x500, double, little-endian).
S3D_CONFIG = {
    "subdir": "SDRBENCH-S3D",
    "field_file": "stat_planar.2.9950E-03.field.d64",
    "shape": (11, 500, 500, 500),
    "var_list": ["CH4", "O2", "CO", "CO2", "H2O", "N2",
                 "Temperature", "Pressure", "VelocityX", "VelocityY", "VelocityZ"],
}


def _prepare_dirs(dataset):
    base = os.path.join(DATA_ROOT, dataset)
    data_dir = os.path.join(base, "data")
    refactor_dir = os.path.join(base, "refactor")
    os.makedirs(data_dir, exist_ok=True)
    os.makedirs(refactor_dir, exist_ok=True)
    return data_dir, refactor_dir


def _make_refactor_slot(refactor_dir, varname):
    os.makedirs(os.path.join(refactor_dir, f"{varname}_refactored"), exist_ok=True)


def _cleanup_source(dataset, subdir):
    """Delete the downloaded archive and the extracted source-format directory
    for `dataset`, keeping only data/ and refactor/. No-op if
    KEEP_SOURCE_ARCHIVES is True."""
    if KEEP_SOURCE_ARCHIVES:
        return
    archive = ARCHIVE_PATHS.get(dataset)
    if archive and os.path.exists(archive):
        os.remove(archive)
        print(f"[{dataset}] removed archive: {archive}")
    src_dir = os.path.join(DATA_ROOT, dataset, subdir)
    if os.path.exists(src_dir):
        shutil.rmtree(src_dir)
        print(f"[{dataset}] removed extracted source dir: {src_dir}")


def rearrange_standard(dataset, cfg):
    data_dir, refactor_dir = _prepare_dirs(dataset)
    src_dir = os.path.join(DATA_ROOT, dataset, cfg["subdir"])
    files = sorted(glob.glob(os.path.join(src_dir, cfg["pattern"])))
    if not files:
        print(f"[{dataset}] WARNING: no files matching {cfg['pattern']} in {src_dir}")
        return
    print(f"[{dataset}] {len(files)} field files -> {data_dir}")
    for path in files:
        varname = cfg["varname"](os.path.basename(path))
        arr = np.fromfile(path, dtype=cfg["src_dtype"]).astype(np.float64)
        arr.tofile(os.path.join(data_dir, f"{varname}.dat"))
        _make_refactor_slot(refactor_dir, varname)
        print(f"    {os.path.basename(path)} -> {varname}.dat  ({arr.size} values)")
    _cleanup_source(dataset, cfg["subdir"])


def rearrange_s3d(cfg):
    data_dir, refactor_dir = _prepare_dirs("S3D")
    field_path = os.path.join(DATA_ROOT, "S3D", cfg["subdir"], cfg["field_file"])
    if not os.path.exists(field_path):
        print(f"[S3D] WARNING: field file not found: {field_path}")
        return
    print(f"[S3D] slicing {cfg['field_file']} {cfg['shape']} -> {data_dir}")
    data = np.fromfile(field_path, dtype=np.float64).reshape(cfg["shape"])
    for idx, varname in enumerate(cfg["var_list"]):
        variable = data[idx, :, :, :]
        variable.tofile(os.path.join(data_dir, f"{varname}.dat"))
        _make_refactor_slot(refactor_dir, varname)
        print(f"    [{idx}] -> {varname}.dat  ({variable.size} values)")
    del data  # release the big in-memory array before deleting its source file
    _cleanup_source("S3D", cfg["subdir"])


for dataset, cfg in REARRANGE.items():
    rearrange_standard(dataset, cfg)
rearrange_s3d(S3D_CONFIG)

print(f"Four datasets downloaded, extracted, rearranged, and source files "
      f"cleaned up under: {DATA_ROOT}")

In [ ]:
# -----------------------------------------------------------------------------
# Per-method command templates, used to tell a reviewer exactly which raw
# command to re-run for a single (dataset, field, method) that looks broken or
# incomplete -- WITHOUT re-running the whole ablation_study.sh script. These
# mirror the ae_scripts/{dataset}_ablation_study.sh templates exactly (same
# binaries, same flags) -- only {data_file}/{refactor_file}/{dims}/{level}/
# {error_bound} are substituted in.
# -----------------------------------------------------------------------------
import os
import re
import math
from collections import defaultdict

import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from scipy import interpolate

# Editable-text fonts for the generated PDFs
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42

# -----------------------------------------------------------------------------
# All result files live under a single root, with the unified naming scheme:
#     ./ae_results/{dataset}_Results/{dataset}_{var}_ablation.txt
# -----------------------------------------------------------------------------
AE_ROOT = "./ae_results"
EB_LINES = (1, 18)          # line range that holds the error-bound sweep
BITS_PER_ELEMENT = 64       # 64-bit double
for _ds in ["CESM", "Miranda", "SCALE", "S3D", "JHTDB"]:
    os.makedirs(f"{AE_ROOT}/{_ds}_Results", exist_ok=True)

# Per-dataset metadata, shared by every figure and table. `variables` uses the
# real field-variable names for each dataset.
DATASETS = {
    "CESM": {
        "prefix": "CESM", "first_var": "CLDICE",
        "variables": ["CLDICE", "CLDLIQ", "CLOUD", "CMFDQR", "CMFDQ", "CMFDT", "CONCLD", "DCQ", "DTCOND", "DTV", "FICE", "GCLDLWP", "ICIMR", "ICLDIWP", "ICLDTWP", "ICWMR", "OMEGAT", "OMEGA", "QC", "QRL", "QRS", "Q", "RELHUM", "T", "UU", "U", "VD01", "VQ", "VT", "VU", "VV", "V", "Z3"],
        "data_size_bytes": 26 * 48 * 1800 * 8,
    },
    "Miranda": {
        "prefix": "Miranda", "first_var": "density",
        "variables": ["density", "diffusivity", "pressure", "velocityx", "velocityy", "velocityz", "viscocity"],
        "data_size_bytes": 256 * 384 * 384 * 8,
    },
    "SCALE": {
        "prefix": "SCALE", "first_var": "PRES",
        "variables": ["PRES", "QC", "QG", "QI", "QR", "QS", "QV", "RH", "U", "V", "W"],
        "data_size_bytes": 98 * 1200 * 1200 * 8,
    },
    "S3D": {
        "prefix": "S3D", "first_var": "CH4",
        "variables": ["CH4", "CO", "CO2", "H2O", "O2", "Temperature", "VelocityX", "VelocityY", "VelocityZ"],
        "data_size_bytes": 500 * 500 * 500 * 8,
    },
}
DATASET_ORDER = ["CESM", "Miranda", "SCALE", "S3D"]

def result_dir(dataset):
    """Unified per-dataset result directory, e.g. ./ae_results/S3D_Results"""
    return f"{AE_ROOT}/{dataset}_Results"

def ablation_file(dataset, var):
    """Unified per-(dataset, variable) ablation file path."""
    return f"{result_dir(dataset)}/{dataset}_{var}_ablation.txt"

# Display names with subscripts (matplotlib mathtext), shared by S3D figures.
var_display = {
    "CH4": r"$\it{CH_4}$",
    "CO": r"$\it{CO}$",
    "CO2": r"$\it{CO_2}$",
    "H2O": r"$\it{H_2O}$",
    "O2": r"$\it{O_2}$",
    "Temperature": r"$\it{Temperature}$",
    "VelocityX": r"$\it{Velocity_X}$",
    "VelocityY": r"$\it{Velocity_Y}$",
    "VelocityZ": r"$\it{Velocity_Z}$",
}

def is_clean_eb(eb, tol=0.01):
    """Keep only error bounds whose mantissa is 1 or 5 (e.g. 1e-3, 5e-4)."""
    if eb <= 0:
        return False
    log_val = math.log10(eb)
    floor_exp = math.floor(log_val)
    mantissa = eb / (10 ** floor_exp)
    return abs(mantissa - 1.0) < tol or abs(mantissa - 5.0) < tol

ABLATION_ERROR_BOUNDS = [0.1, 0.05, 0.01, 0.005, 0.001, 0.0005, 0.0001, 0.00005,
                         0.00001, 0.000005, 0.000001, 0.0000005, 0.0000001,
                         0.00000005, 0.00000001, 0.000000005, 0.000000001]

ABLATION_DIMS = {
    "CESM":    {"dims": "26 1800 3600",   "level": 3},
    "Miranda": {"dims": "256 384 384",    "level": 4},
    "SCALE":   {"dims": "98 1200 1200",   "level": 4},
    "S3D":     {"dims": "500 500 500",    "level": 4},
}

# method_num -> (display name, refactor command template, reconstruct command
# template). Templates use .format(data_file=..., refactor_file=..., dims=...,
# level=..., error_bound=...).
METHOD_COMMANDS = {
    1: ("IPComp",
        './external/IPComp/build/src/refactor {data_file} -d -3 {dims} {refactor_file}',
        './external/IPComp/build/src/reconstructor {data_file} -d -3 {dims} -1 {error_bound} {refactor_file}'),
    2: ("PMGARD-EB",
        './build/test/test_mdr_refactor {data_file} {level} 60 3 {dims} {refactor_file} 1',
        './build/test/test_mdr_reconstructor {data_file} 1 {error_bound} {refactor_file} 1'),
    3: ("PMGARD-PSNR",
        './build/test/test_mdr_refactor {data_file} {level} 60 3 {dims} {refactor_file} 0',
        './build/test/test_mdr_reconstructor {data_file} 1 {error_bound} {refactor_file} 0'),
    4: ("SZ3-R",
        './build/test/test_PSZ3-delta_refactor {data_file} {refactor_file} 18 3 {dims} -d',
        './build/test/test_PSZ3-delta_reconstructor {data_file} {refactor_file} 1 {error_bound} -d'),
    5: ("AdatInterp-EB",
        './build/test/two_modes_refactor {data_file} -d {level} 60 3 {dims} {refactor_file} -PerBit -eb -no_CP',
        './build/test/two_modes_reconstructor {data_file} -d 1 {error_bound} {refactor_file} -PerBit -DP -no_CP'),
    6: ("CoeffDecom-EB",
        './build/test/two_modes_refactor {data_file} -d {level} 60 3 {dims} {refactor_file} -PerBit -eb -CP',
        './build/test/two_modes_reconstructor {data_file} -d 1 {error_bound} {refactor_file} -PerBit -DP -CP'),
    7: ("AdatInterp-PSNR",
        './build/test/two_modes_refactor {data_file} -d {level} 60 3 {dims} {refactor_file} -XOR -PSNR -no_CP',
        './build/test/two_modes_reconstructor {data_file} -d 1 {error_bound} {refactor_file} -XOR -DP -no_CP'),
    8: ("CoeffDecom-PSNR",
        './build/test/two_modes_refactor {data_file} -d {level} 60 3 {dims} {refactor_file} -XOR -PSNR -CP',
        './build/test/two_modes_reconstructor {data_file} -d 1 {error_bound} {refactor_file} -XOR -DP -CP'),
}


def _dataset_var_paths(dataset, var):
    """Same path convention as ae_scripts/*_ablation_study.sh: DATA_ROOT must
    be set earlier in the notebook (Part 1 configuration)."""
    data_dict_path = os.path.join(DATA_ROOT, dataset)
    data_file = f"{data_dict_path}/data/{var}.dat"
    refactor_file = f"{data_dict_path}/refactor/{var}_refactored"
    return data_file, refactor_file


def print_fix_command(dataset, var, method_num, missing_ebs=None):
    """Print the bare command(s) to re-run for ONE (dataset, field, method)
    that looks broken/incomplete. Run these yourself from the ProDM directory,
    read the printed PSNR/Bitrate/time from the terminal, and manually update
    the corresponding "Method#{method_num}, ..." line(s) in:
        {ablation_file(dataset, var)}
    """
    method_num = int(method_num)
    if dataset not in ABLATION_DIMS or method_num not in METHOD_COMMANDS:
        print(f"  (no command template for {dataset} method#{method_num})")
        return

    name, refactor_tmpl, reconstruct_tmpl = METHOD_COMMANDS[method_num]
    dims = ABLATION_DIMS[dataset]["dims"]
    level = ABLATION_DIMS[dataset]["level"]
    data_file, refactor_file = _dataset_var_paths(dataset, var)

    print(f"[{dataset}/{var}] Method#{method_num} ({name}) looks incomplete. "
          f"Update: {ablation_file(dataset, var)}")
    print(f"  Refactor (only needed if the 'Refactor time' line is also missing):")
    print("   ", refactor_tmpl.format(data_file=data_file, refactor_file=refactor_file,
                                      dims=dims, level=level))

    ebs = missing_ebs if missing_ebs else ABLATION_ERROR_BOUNDS
    print(f"  Reconstruct for error bound(s) {ebs}:")
    for eb in ebs:
        cmd = reconstruct_tmpl.format(data_file=data_file, refactor_file=refactor_file,
                                      dims=dims, level=level, error_bound=eb)
        print("   ", cmd)
    print()


# -----------------------------------------------------------------------------
# Shared loader used by the baseline-comparison figures (Fig 9/10 and zoom-ins).
# Reads every variable's ablation file for one dataset, filters to clean error
# bounds, then aggregates per method: mean bitrate and combined PSNR across
# all fields. Returns (error_bounds, best_curves_prog, best_PSNR).
# -----------------------------------------------------------------------------
def load_dataset_filtered(dataset):
    meta = DATASETS[dataset]
    variables, first_var = meta["variables"], meta["first_var"]

    # Collects (var, method, eb, detail) problems. method/eb are None when the
    # problem isn't tied to one specific method/error-bound (e.g. a missing
    # file, or a count-mismatch that could affect any error bound).
    problems = []

    def _need(fname):
        path = ablation_file(dataset, fname)
        if not os.path.exists(path):
            problems.append((fname, None, None, f"result file not found: {path}"))
            return None
        return open(path, 'r').readlines()

    def _fail():
        print(f"[{dataset}] found {len(problems)} problem(s) while parsing ablation results:")
        for var, method, eb, detail in problems:
            tag = f"method#{method}" if method is not None else "(whole file)"
            print(f"  - dataset={dataset}, field={var}, {tag}: {detail}")
        print()

        # Group by (var, method), collecting known missing eb's if ALL
        # instances for that (var, method) have a known eb; otherwise fall
        # back to listing every error bound.
        grouped = defaultdict(list)
        for var, method, eb, detail in problems:
            if method is not None:
                grouped[(var, method)].append(eb)
        for (var, method), ebs in grouped.items():
            missing_ebs = ebs if all(e is not None for e in ebs) else None
            print_fix_command(dataset, var, method, missing_ebs)

        # Problems with no method at all (missing file / malformed line)
        no_method = [(var, detail) for var, method, eb, detail in problems if method is None]
        for var, detail in no_method:
            print(f"[{dataset}/{var}] {detail}")
            print(f"  Easiest fix: re-run the whole field:")
            print(f"    bash ./ae_scripts/{dataset}_ablation_study.sh {DATA_ROOT}/{dataset} {var}")
            print()

        raise RuntimeError(
            f"[{dataset}] {len(problems)} problem(s) found -- see the commands "
            f"printed above. After running them and updating the corresponding "
            f".txt file(s) by hand, re-run this cell."
        )

    # --- error-bound axis, read from the first variable's file ----------------
    txt = _need(first_var)
    if txt is None:
        _fail()
    lo, hi = EB_LINES
    if len(txt) < hi:
        problems.append((first_var, None, None,
                         f"expected at least {hi} lines to read error bounds "
                         f"(EB_LINES={EB_LINES}), but file has {len(txt)}"))
        _fail()

    all_eb = []
    for i in range(lo, hi):
        line = txt[i].rstrip()
        if "ErrorBound=" not in line:
            problems.append((first_var, None, None, f"line {i}: missing 'ErrorBound=' -> {line!r}"))
            continue
        all_eb.append(float(line.split("ErrorBound=")[1].split(",")[0]))
    if problems:
        _fail()
    error_bounds = [eb for eb in all_eb if is_clean_eb(eb)]

    # --- per-variable curves --------------------------------------------------
    curves_progs = defaultdict(lambda: defaultdict(list))
    NRMSEs = defaultdict(lambda: defaultdict(list))

    for var in variables:
        lines = _need(var)
        if lines is None:
            continue
        for ln_no, line in enumerate(lines):
            if "Refactor" in line:
                continue
            line_s = line.rstrip()

            method = None
            if "#" in line_s.split(",")[0]:
                try:
                    method = line_s.split(",")[0].split("#")[1]
                except ValueError:
                    method = None

            if "ErrorBound=" not in line_s:
                problems.append((var, method, None, f"line {ln_no}: missing 'ErrorBound=' -> {line_s!r}"))
                continue
            eb = float(line_s.split("ErrorBound=")[1].split(",")[0])
            if eb not in all_eb or not is_clean_eb(eb):
                continue

            if method is None:
                problems.append((var, None, eb, f"line {ln_no}: missing 'Method#' -> {line_s!r}"))
                continue

            if "Bitrate = " not in line_s:
                problems.append((var, method, eb, f"line {ln_no}: missing 'Bitrate = ' -> {line_s!r}"))
                continue
            bitrate = float(line_s.split("Bitrate = ")[1].split(",")[0])

            NRMSE = None
            if "NRMSE = " in line_s:
                NRMSE = float(line_s.split("NRMSE = ")[1].split(",")[0])
            elif "NRMSE=" in line_s:
                NRMSE = float(line_s.split("NRMSE= ")[1].split(",")[0])
            if NRMSE is None:
                problems.append((var, method, eb, f"line {ln_no}: missing 'NRMSE' -> {line_s!r}"))
                continue

            curves_progs[var][method].append(bitrate)
            NRMSEs[var][method].append(NRMSE)

    if problems:
        _fail()

    # --- consistency check: every var/method must cover all error bounds ------
    n_eb = len(error_bounds)
    ref_methods = set(curves_progs[variables[0]].keys())
    for var in variables:
        for method in ref_methods:
            got = len(curves_progs[var].get(method, []))
            if got != n_eb:
                problems.append((var, method, None,
                    f"method #{method}: got {got} data point(s) but expected "
                    f"{n_eb} (one per clean error bound). Incomplete/failed run."))
    if problems:
        _fail()

    # --- aggregate across variables -------------------------------------------
    best_curves_prog = defaultdict(list)
    best_PSNR = defaultdict(list)
    for method in curves_progs[variables[0]].keys():
        for eb_i in range(n_eb):
            sum_br, sum_nrmse_sq, n_br, n_nrmse = 0, 0, 0, 0
            for var in variables:
                if curves_progs[var][method][eb_i] is None or NRMSEs[var][method][eb_i] is None:
                    continue
                n_br += 1; n_nrmse += 1
                sum_br += curves_progs[var][method][eb_i]
                sum_nrmse_sq += NRMSEs[var][method][eb_i] ** 2
            best_curves_prog[method].append(sum_br / n_br)
            best_PSNR[method].append(20 * math.log10(math.sqrt(n_nrmse) / math.sqrt(sum_nrmse_sq)))
    return error_bounds, best_curves_prog, best_PSNR

In [ ]:
# =============================================================================
# Submit cluster jobs from the notebook.
#
# Run this notebook from the ProDM directory (as noted above). Defines
# submit_job(), which dispatches to one of two execution paths depending on
# the EXECUTION_MODE set earlier:
#
#   "slurm"    — read the template .sh in ./ae_scripts, inject your
#                cluster-specific SBATCH_SETTINGS into the #SBATCH directives,
#                write a filled copy, and submit it with sbatch. Each job
#                receives the dataset path as an argument ($1).
#
#   "parallel" — no scheduler involved: append a plain bash command (running
#                the same template .sh directly, no #SBATCH processing) to a
#                job queue file. Nothing actually runs at this point -- call
#                run_queued_jobs() afterwards to execute the whole queue with
#                GNU Parallel, bounded by MAX_PARALLEL_JOBS concurrent workers.
#
# submit_job()'s signature is identical either way, so every call site below
# in this notebook works unchanged regardless of which mode you picked.
# =============================================================================
import os
import subprocess
import os
import subprocess


def _inline_comment(line):
    """Return the trailing '  # ...' part of an #SBATCH line, or '' if none."""
    idx = line.find("#", len("#SBATCH"))
    if idx == -1:
        return ""
    return "  " + line[idx:].rstrip("\n")


def _format_directive(key, val):
    """Render one #SBATCH directive. '--flag' style uses '=', '-x' style uses a
    space. A flag with no value (val is '' or whitespace) is emitted bare."""
    if val is None or str(val).strip() == "":
        return f"#SBATCH {key}"
    if key.startswith("--"):
        return f"#SBATCH {key}={val}"
    return f"#SBATCH {key} {val}"


def fill_sbatch(template_path, settings):
    """Return the script text with #SBATCH directives filled in.

    - Options present in the template are replaced in place (both `--key=` and
      `-k <val>` styles), preserving any trailing inline comment.
    - Options in `settings` that are NOT in the template are appended right
      after the last #SBATCH line (so you can add --constraint, --exclusive,
      --gres, etc. without editing the template).
    - If --mail-user is empty, the --mail-* lines are removed.
    """
    out_lines = []
    drop_mail = not settings.get("--mail-user")
    used_keys = set()          # settings keys that matched a template line
    last_sbatch_out_idx = None # index in out_lines just after the last #SBATCH

    for line in open(template_path):
        stripped = line.strip()

        if drop_mail and stripped.startswith("#SBATCH --mail-"):
            continue

        replaced = False
        if stripped.startswith("#SBATCH "):
            for key, val in settings.items():
                if str(val).strip() == "" and not stripped.startswith(f"#SBATCH {key}"):
                    continue
                eq_prefix = f"#SBATCH {key}="
                sp_prefix = f"#SBATCH {key} "
                bare = f"#SBATCH {key}"
                if (stripped.startswith(eq_prefix) or stripped.startswith(sp_prefix)
                        or stripped == bare):
                    out_lines.append(_format_directive(key, val) + _inline_comment(line) + "\n")
                    used_keys.add(key)
                    replaced = True
                    break

        if not replaced:
            out_lines.append(line)

        if out_lines and out_lines[-1].strip().startswith("#SBATCH"):
            last_sbatch_out_idx = len(out_lines)

    extras = []
    for key, val in settings.items():
        if key in used_keys:
            continue
        if val is None or (str(val) == ""):
            continue
        extras.append(_format_directive(key, val) + "\n")

    if extras and last_sbatch_out_idx is not None:
        out_lines[last_sbatch_out_idx:last_sbatch_out_idx] = extras
    elif extras:
        insert_at = 1 if out_lines and out_lines[0].startswith("#!") else 0
        out_lines[insert_at:insert_at] = extras

    return "".join(out_lines)


def replace_module_lines(script_text, module_setup):
    """Replace the block of consecutive 'module ...' lines with module_setup.

    If the template has none, module_setup is inserted after the last #SBATCH
    line. If module_setup is empty, existing module lines are removed.
    """
    lines = script_text.split("\n")
    out = []
    i = 0
    n = len(lines)
    replaced = False
    inserted_after_sbatch_idx = None

    while i < n:
        stripped = lines[i].strip()
        if stripped.startswith("module ") or stripped == "module purge":
            while i < n and (lines[i].strip().startswith("module ")
                             or lines[i].strip() == "module purge"):
                i += 1
            if not replaced:
                if module_setup.strip():
                    out.extend(module_setup.split("\n"))
                replaced = True
            continue
        out.append(lines[i])
        if lines[i].strip().startswith("#SBATCH"):
            inserted_after_sbatch_idx = len(out)
        i += 1

    if not replaced and module_setup.strip() and inserted_after_sbatch_idx is not None:
        out[inserted_after_sbatch_idx:inserted_after_sbatch_idx] = ["", *module_setup.split("\n")]

    return "\n".join(out)


def _submit_job_slurm(script_name, data_path=None, extra_args=None, module_setup=None,
                      sbatch_settings=None):
    """Original SLURM path: fill #SBATCH directives and sbatch the script."""
    settings = sbatch_settings if sbatch_settings is not None else SBATCH_SETTINGS
    if not settings.get("--partition") or not settings.get("-A"):
        raise SystemExit("Please set at least --partition and -A in the SBATCH settings.")

    template = os.path.join(AE_SCRIPTS, script_name)
    if not os.path.exists(template):
        print(f"[{script_name}]  ERROR: template not found at {template}")
        return None

    filled_dir = os.path.join(AE_SCRIPTS, "_filled")
    os.makedirs(filled_dir, exist_ok=True)
    filled = os.path.join(filled_dir, script_name)

    text = fill_sbatch(template, settings)
    if module_setup is not None:
        text = replace_module_lines(text, module_setup)

    with open(filled, "w") as f:
        f.write(text)
    os.chmod(filled, 0o755)

    cmd = ["sbatch", filled]
    if data_path is not None:
        cmd.append(data_path)
    if extra_args:
        cmd.extend(extra_args)

    result = subprocess.run(cmd, capture_output=True, text=True)
    print(f"[{script_name}]  args={[data_path, *(extra_args or [])]}")
    if result.stdout:
        print("  ", result.stdout.strip())
    if result.returncode != 0:
        print("   ERROR:", result.stderr.strip())
    return result


def _submit_job_parallel(script_name, data_path=None, extra_args=None, module_setup=None,
                         sbatch_settings=None):
    """Parallel path: queue a plain bash command, run later via run_queued_jobs().
    module_setup / sbatch_settings are accepted for signature compatibility but
    ignored -- no scheduler, no module system on a single node.
    """
    template = os.path.join(AE_SCRIPTS, script_name)
    if not os.path.exists(template):
        print(f"[{script_name}]  ERROR: template not found at {template}")
        return None

    parts = ["bash", template]
    if data_path is not None:
        parts.append(str(data_path))
    if extra_args:
        parts.extend(str(a) for a in extra_args)

    line = " ".join(f'"{p}"' if " " in p else p for p in parts)

    with open(JOBLIST_PATH, "a") as f:
        f.write(line + "\n")

    print(f"Queued: {line}")
    return line


def submit_job(script_name, data_path=None, extra_args=None, module_setup=None,
               sbatch_settings=None):
    """Dispatch to the SLURM or parallel implementation based on EXECUTION_MODE."""
    if EXECUTION_MODE == "slurm":
        return _submit_job_slurm(script_name, data_path, extra_args, module_setup,
                                 sbatch_settings)
    elif EXECUTION_MODE == "parallel":
        return _submit_job_parallel(script_name, data_path, extra_args, module_setup,
                                    sbatch_settings)
    else:
        raise ValueError(f"Unknown EXECUTION_MODE: {EXECUTION_MODE!r}")


def reset_job_queue():
    """Clear the parallel job queue. Call this before queuing a fresh batch."""
    open(JOBLIST_PATH, "w").close()
    if os.path.exists(JOBLOG_PATH):
        os.remove(JOBLOG_PATH)
    print(f"Cleared {JOBLIST_PATH}")


def run_queued_jobs(jobs=MAX_PARALLEL_JOBS):
    """Run everything currently queued in JOBLIST_PATH with GNU Parallel,
    blocking until all jobs finish. Only relevant when EXECUTION_MODE == "parallel".

    Check run.log afterwards for per-job exit codes/timing, or retry failures:
        parallel --joblog run.log --resume-failed < joblist.txt
    """
    if EXECUTION_MODE != "parallel":
        print(f"EXECUTION_MODE is {EXECUTION_MODE!r}, nothing to run here "
              f"(jobs were already submitted via sbatch).")
        return None

    if not os.path.exists(JOBLIST_PATH) or os.path.getsize(JOBLIST_PATH) == 0:
        print("Job queue is empty -- nothing to run.")
        return None

    with open(JOBLIST_PATH) as f:
        n_jobs = sum(1 for _ in f)
    print(f"Running {n_jobs} queued job(s) with up to {jobs} in parallel...")

    with open(JOBLIST_PATH) as f:
        result = subprocess.run(
            ["parallel", "-j", str(jobs), "--joblog", JOBLOG_PATH, "--eta"],
            stdin=f,
            capture_output=True, text=True,
        )

    print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if result.returncode == 0:
        print("All queued jobs finished successfully.")
    else:
        print(f"Some jobs failed (parallel exit code {result.returncode}). "
              f"Check {JOBLOG_PATH} for details, e.g.:\n"
              f"  awk -F'\\t' '$7!=0' {JOBLOG_PATH}")

    reset_job_queue()
    return result


if EXECUTION_MODE == "parallel":
    reset_job_queue()

### Job Submission — S3D decomposition & encoding timing (Fig. 6)

After downloading and rearrangement of the above four datasets, move on to submit jobs.

**All scripts are listed in "./ae_scripts".**

First, submit *"./ae_scripts/S3D_decompose_time.sh"* and *"./ae_scripts/S3D_encoding_time.sh"*. Wait until they are done.

How you submit these depends on the `EXECUTION_MODE` you set earlier:

**If `EXECUTION_MODE = "slurm"`:**

You will need to adjust the **SBATCH directives** (the `#SBATCH` lines at the top of each script) to match your cluster before submitting. In particular, fill in the following, which are left blank in the templates:
- `--partition=` — the partition/queue on your cluster
- `-A` — your project allocation account
- `--mail-user=` — your email address (optional; or remove the two `--mail-*` lines)

You may also want to review `--time`, `--nodes`, and `--ntasks` and adjust them to your allocation.

Submit from the **ProDM** directory (make sure your shell's working directory is `ProDM/`), e.g.:

    cd ProDM
    sbatch ./ae_scripts/S3D_decompose_time.sh
    sbatch ./ae_scripts/S3D_encoding_time.sh

Check job status with `squeue -u $USER`.

**If `EXECUTION_MODE = "parallel"` (e.g. a single machine/bare-metal node without SLURM):**

No `#SBATCH` settings needed — the scripts run directly as plain bash, with the `#SBATCH` lines simply ignored as comments. Submit from the **ProDM** directory:

    cd ProDM
    bash ./ae_scripts/S3D_decompose_time.sh
    bash ./ae_scripts/S3D_encoding_time.sh

These two are quick enough to just run directly like this; no need to queue them.

> Either way, you can also use the `submit_job()` cell provided later in this notebook instead of running these commands by hand — it dispatches to the right path automatically based on `EXECUTION_MODE`. Both approaches work; pick whichever you prefer.

In [ ]:
# -----------------------------------------------------------------------------
# Step 1: submit the two S3D timing jobs. Both take the S3D data path as $1.
# -----------------------------------------------------------------------------
s3d_path = os.path.join(DATA_ROOT, "S3D")
for job in ["S3D_decompose_time.sh", "S3D_encoding_time.sh"]:
    submit_job(job, data_path=s3d_path)

if EXECUTION_MODE == "parallel":
    print("\n[parallel] Timing jobs running...\
           \n Around 7 minutes on a single compute_gigaio node (CHI@UC, dual EPYC 7763)")
    run_queued_jobs()
    print("\n[parallel] Timing jobs complete. ")
else:
    print("\n[SLURM] Submitted. Check status with:  squeue -u $USER. Please wait till they are done.\
           \n Around 7 minutes.")


### Fig. 6 — Throughput of decomposition and encoding (S3D)

Throughput of decomposition and generic bit-plane encoding before and after optimization on the S3D dataset with 9 valid fields.

In [ ]:
decompose_file = f"{result_dir('S3D')}/S3D_decompose_time.txt"
encoding_file = f"{result_dir('S3D')}/S3D_encoding_time.txt"

data_size_bytes = 500 * 500 * 500 * 8  # bytes per variable

# Rename mapping
rename_map = {
    'PMGARD': 'PMGARD Interp',
    'Fast': 'Fastest Direct Interp.',
    'old perbit': 'PMGARD generic',
    'perbit': 'Optimized generic',
}

var_display = {
    "CH4": r"$\it{CH_4}$",
    "CO": r"$\it{CO}$",
    "CO2": r"$\it{CO_2}$",
    "H2O": r"$\it{H_2O}$",
    "O2": r"$\it{O_2}$",
    "Temperature": r"$\it{Temp.}$",
    "VelocityX": r"$\it{Vel._X}$",
    "VelocityY": r"$\it{Vel._Y}$",
    "VelocityZ": r"$\it{Vel._Z}$",
}

def parse_file(filepath):
    """Parse time file, return dict: {method_name: {variable: time}}"""
    results = {}
    with open(filepath, 'r') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            m = re.match(
                r'Method#\d+,\s*(.+?),\s*Variable=(\w+),\s*\w+ time:\s*([\d.]+)',
                line
            )
            if m:
                method_name = m.group(1).strip()
                method_name = rename_map.get(method_name, method_name)
                variable = m.group(2).strip()
                time_val = float(m.group(3))
                if method_name not in results:
                    results[method_name] = {}
                results[method_name][variable] = time_val
    return results

decompose_data = parse_file(decompose_file)
encoding_data = parse_file(encoding_file)

def compute_throughput(data_dict):
    """For each method, compute throughput (GB/s) per variable"""
    out = {}
    for method, var_times in data_dict.items():
        variables = sorted(var_times.keys())
        tp = np.array([data_size_bytes / var_times[v] / 1e9 for v in variables])
        out[method] = (variables, tp)
    return out

dec_tp = compute_throughput(decompose_data)
enc_tp = compute_throughput(encoding_data)

# --- Plot ---
fig, axes = plt.subplots(1, 2, figsize=(6, 2.5))
bar_width = 0.35
colors = ['#4998C7', '#eebe42']

for ax, tp_dict, title in zip(axes, [dec_tp, enc_tp], ['Decomposition', 'Encoding']):
    methods = list(tp_dict.keys())
    variables, tp0 = tp_dict[methods[0]]
    _, tp1 = tp_dict[methods[1]]
    x = np.arange(len(variables))

    ax.bar(x - bar_width / 2, tp0, bar_width, label=methods[0], color=colors[0])
    ax.bar(x + bar_width / 2, tp1, bar_width, label=methods[1], color=colors[1])

    ax.set_xlabel('Field', fontsize=11)
    ax.set_ylabel('Throughput (GB/s)', fontsize=11)
    ax.set_title(f'{title} Throughput', fontsize=12)
    ax.set_xticks(x)
    xlabels = [var_display.get(v, v) for v in variables]
    ax.set_xticklabels(xlabels, rotation=45, ha='right', fontsize=9)
    ax.tick_params(axis='y', labelsize=10)
    ax.legend(fontsize=7, loc='upper right', ncol=1, frameon=True)

plt.tight_layout()
# plt.savefig("S3D_throughput.pdf", bbox_inches="tight")
plt.show()

### Job Submission — Ablation study (Fig. 7–10, Table 2–3)

Then submit the ablation-study jobs for Fig. 7–10 and Table 2-3. How you run
these depends on the `EXECUTION_MODE` you set earlier.

**If `EXECUTION_MODE = "slurm"`:**

You can either run the shell chain:

    sh ./ae_scripts/all_sbatch.sh {DATA_ROOT}

which fans out to `submit_{CESM,Miranda,SCALE,S3D}.sh`, each looping over that
dataset's variables and calling `sbatch {dataset}_ablation_study.sh {DATA_ROOT}/{dataset} {var}`.
Only the `*_ablation_study.sh` scripts carry `#SBATCH` directives, so those are
the ones whose headers you need to fill in (same fields as before).

Alternatively, use the job-submission cell below, which fills the SBATCH headers
from your `SBATCH_SETTINGS` and submits the same set of jobs for you.

Wait until all jobs finish (`squeue -u $USER`) before running the figure/table cells.

**If `EXECUTION_MODE = "parallel"` (e.g. a single machine/bare-metal node without SLURM):**

Use the job-submission cell below (`submit_job(...)` calls for each dataset's
variables), which queues all 63 jobs — CESM (33), Miranda (7), SCALE (12),
S3D (11) — instead of submitting them via sbatch. Nothing runs yet at that
point; after queuing them all, run the `run_queued_jobs()` cell to execute the
full batch with GNU Parallel, bounded by `MAX_PARALLEL_JOBS` concurrent workers.

The shell chain (`all_sbatch.sh`) is SLURM-only (it calls `sbatch` directly) and
should not be used in this mode.

Wait for `run_queued_jobs()` to finish (it blocks until all jobs are done) before
running the figure/table cells.

In [ ]:
# -----------------------------------------------------------------------------
# Step 2: submit the ablation-study jobs for Fig. 7-10 (CESM, Miranda, SCALE,
# S3D). This mirrors the shell chain:
#
#     all_sbatch.sh {DATA_ROOT}
#        -> submit_CESM.sh    {DATA_ROOT}/CESM    -> CESM_ablation_study.sh   {DATA_ROOT}/CESM  {var}
#        -> submit_Miranda.sh {DATA_ROOT}/Miranda -> Miranda_ablation_study.sh {DATA_ROOT}/Miranda {var}
#        -> submit_SCALE.sh   {DATA_ROOT}/SCALE   -> SCALE_ablation_study.sh  {DATA_ROOT}/SCALE {var}
#        -> submit_S3D.sh     {DATA_ROOT}/S3D     -> S3D_ablation_study.sh    {DATA_ROOT}/S3D   {var}
#
# Only the *_ablation_study.sh templates carry #SBATCH directives, so those are
# the ones that get filled. One job is submitted per (dataset, variable):
# {DATA_ROOT}/{dataset} is passed as $1 and the variable name as $2.
# The variable lists come from the shared DATASETS config.
# -----------------------------------------------------------------------------
    
def submit_ablation_study():
    for ds_name in DATASET_ORDER:
        ds_path = os.path.join(DATA_ROOT, ds_name)
        variables = DATASETS[ds_name]["variables"]
        script = f"{ds_name}_ablation_study.sh"
        print(f"\n=== {ds_name}: submitting {len(variables)} jobs "
              f"({script}, arg={ds_path}) ===")
        for var in variables:
            submit_job(script, data_path=ds_path, extra_args=[var])


submit_ablation_study()

if EXECUTION_MODE == "parallel":
    print("\n[parallel] Ablation jobs running...\
           \n Around 100 minutes on a single compute_gigaio node (CHI@UC, dual EPYC 7763)")
    run_queued_jobs()
    print("\n[parallel] Ablation jobs complete. "
          "On a single compute_gigaio node (CHI@UC, dual EPYC 7763) this takes "
          "about 100 minutes.")
else:
    print("\n[SLURM] All ablation-study jobs submitted. "
          "Check status with: squeue -u $USER, and wait until they finish.\n"
          "On recent CPUs, with all fields scheduled concurrently, the per-dataset "
          "runtime is approximately: Miranda ~6 min, S3D ~15 min, SCALE ~17 min, "
          "CESM ~25 min (≈30 min total if fully concurrent).")

### Fig. 7 — Ablation study, error-bound mode (S3D)

Ablation study of error-bound mode on the S3D dataset.

In [ ]:
cfg = {"xlim_eb": (0, 8)}

# Same ablation files as every figure; this plot uses method indices 2/5/6.
method_dict = {"2": "PMGARD", "5": "AdatInterp", "6": "CoeffDecom"}
method_colors = {"2": "brown", "5": "blue", "6": "red"}
method_ls = {"2": "--", "5": "--", "6": "-"}
plot_methods = ["2", "5", "6"]

# Load data per variable
first_file = ablation_file("S3D", DATASETS["S3D"]["first_var"])
txt = open(first_file, 'r').readlines()
lo, hi = EB_LINES
all_eb = [float(txt[i].rstrip().split("ErrorBound=")[1].split(",")[0]) for i in range(lo, hi)]
error_bounds = [eb for eb in all_eb if is_clean_eb(eb)]

per_var_bitrate = defaultdict(lambda: defaultdict(list))

for var in DATASETS["S3D"]["variables"]:
    filename = ablation_file("S3D", var)
    for line in open(filename, 'r').readlines():
        if "Refactor" in line:
            continue
        eb = float(line.rstrip().split("ErrorBound=")[1].split(",")[0])
        if eb not in all_eb or not is_clean_eb(eb):
            continue
        method = line.rstrip().split(",")[0].split("#")[1]
        bitrate = float(line.rstrip().split("Bitrate = ")[1].split(",")[0])
        per_var_bitrate[var][method].append(bitrate)

# Plot: 3x3 grid for 9 variables
variables = DATASETS["S3D"]["variables"]
n_vars = len(variables)
ncols = 3
nrows = math.ceil(n_vars / ncols)

fig, axes = plt.subplots(nrows, ncols, figsize=(6, nrows * 1.6),
                         sharex=True, sharey=True)
axes_flat = axes.flatten()

for i, var in enumerate(variables):
    ax = axes_flat[i]
    for method in plot_methods:
        if method not in per_var_bitrate[var]:
            continue
        ax.plot(per_var_bitrate[var][method], error_bounds, linestyle=method_ls[method],
                color=method_colors[method], marker='.', label=method_dict[method],
                linewidth=1, markersize=4)
    ax.set_xlim(cfg["xlim_eb"])
    ax.set_ylim(5e-8, 5e-1)
    ax.set_yscale('log')
    ax.set_title(var_display.get(var, var), fontsize=10)
    ax.tick_params(labelsize=7)

# Hide unused subplots
for j in range(n_vars, len(axes_flat)):
    axes_flat[j].set_visible(False)

# Shared axis labels
fig.supxlabel('Bitrate', fontsize=11)
fig.supylabel('Error Bound', fontsize=11)

handles, labels = axes_flat[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=len(plot_methods), bbox_to_anchor=(0.5, -0.06), fontsize=9)
plt.tight_layout()
plt.subplots_adjust(bottom=0.12, left=0.1)
# plt.savefig('S3D_per_field_ablation.pdf', bbox_inches='tight')
plt.show()

### Fig. 8 — Ablation study, PSNR mode (S3D)

Ablation study of PSNR mode on the S3D dataset.

In [ ]:
cfg = {"xlim_psnr": (0, 8), "ylim_psnr": (60, 180)}

# Same ablation files as every figure; this plot uses method indices 1/7/8.
method_dict = {"1": "IPComp", "7": "AdatInterp", "8": "CoeffDecom"}
method_colors = {"1": "cyan", "7": "blue", "8": "red"}
method_ls = {"1": "--", "7": "--", "8": "-"}
plot_methods = ["1", "7", "8"]


# Load data per variable
first_file = ablation_file("S3D", DATASETS["S3D"]["first_var"])
txt = open(first_file, 'r').readlines()
lo, hi = EB_LINES
all_eb = [float(txt[i].rstrip().split("ErrorBound=")[1].split(",")[0]) for i in range(lo, hi)]
error_bounds = [eb for eb in all_eb if is_clean_eb(eb)]

per_var_bitrate = defaultdict(lambda: defaultdict(list))
per_var_psnr = defaultdict(lambda: defaultdict(list))

for var in DATASETS["S3D"]["variables"]:
    filename = ablation_file("S3D", var)
    for line in open(filename, 'r').readlines():
        if "Refactor" in line:
            continue
        eb = float(line.rstrip().split("ErrorBound=")[1].split(",")[0])
        if eb not in all_eb or not is_clean_eb(eb):
            continue
        method = line.rstrip().split(",")[0].split("#")[1]
        bitrate = float(line.rstrip().split("Bitrate = ")[1].split(",")[0])
        PSNR = None
        if "PSNR = " in line:
            PSNR = float(line.rstrip().split("PSNR = ")[1].split(",")[0])
        if "PSNR=" in line:
            PSNR = float(line.rstrip().split("PSNR= ")[1].split(",")[0])
        per_var_bitrate[var][method].append(bitrate)
        per_var_psnr[var][method].append(PSNR)

# Plot: 3x3 grid for 9 variables
variables = DATASETS["S3D"]["variables"]
n_vars = len(variables)
ncols = 3
nrows = math.ceil(n_vars / ncols)

fig, axes = plt.subplots(nrows, ncols, figsize=(6, nrows * 1.6),
                         sharex=True, sharey=True)
axes_flat = axes.flatten()

for i, var in enumerate(variables):
    ax = axes_flat[i]
    for method in plot_methods:
        if method not in per_var_bitrate[var]:
            continue
        br = per_var_bitrate[var][method]
        psnr = per_var_psnr[var][method]
        valid = [(b, p) for b, p in zip(br, psnr) if p is not None]
        if not valid:
            continue
        br_v, psnr_v = zip(*valid)
        ax.plot(br_v, psnr_v, linestyle=method_ls[method],
                color=method_colors[method], marker='.', label=method_dict[method],
                linewidth=1, markersize=4)
    ax.set_xlim(cfg["xlim_psnr"])
    ax.set_ylim(cfg["ylim_psnr"])
    ax.set_title(var_display.get(var, var), fontsize=10)
    ax.tick_params(labelsize=7)

# Hide unused subplots
for j in range(n_vars, len(axes_flat)):
    axes_flat[j].set_visible(False)

# Shared axis labels
fig.supxlabel('Bitrate', fontsize=11)
fig.supylabel('PSNR', fontsize=11)

handles, labels = axes_flat[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=len(plot_methods), bbox_to_anchor=(0.5, -0.06), fontsize=9)
plt.tight_layout()
plt.subplots_adjust(bottom=0.12, left=0.1)
# plt.savefig('S3D_per_field_PSNR_ablation.pdf', bbox_inches='tight')
plt.show()

### Fig. 9 — Baseline comparison, error-bound mode

Baseline comparison of error-bound mode.

In [ ]:
# Per-figure axis limits (shared dataset metadata comes from DATASETS).
AXES = {ds: {"xlim_eb": (0, 8), "ylim_eb": (5e-8, 5e-1)} for ds in DATASET_ORDER}

# Same ablation files as every figure; this plot uses indices 1/2/4/6 (6 = Ours).
method_dict = {"1": "IPComp", "2": "PMGARD", "4": "SZ3-R", "6": "Ours"}
method_colors = {"1": "purple", "2": "brown", "4": "cyan", "6": "red"}

eb_plot_methods = ["1", "2", "4", "6"]
eb_baseline_methods = ["1", "2", "4"]
cp_method = "6"

# ===== EB vs Bitrate =====
fig, axes = plt.subplots(2, 2, figsize=(6, 4.5))

for idx, ds_name in enumerate(DATASET_ORDER):
    ax = axes[idx // 2][idx % 2]
    lim = AXES[ds_name]
    error_bounds, best_curves_prog, best_PSNR = load_dataset_filtered(ds_name)

    for method in sorted(best_curves_prog.keys()):
        if method not in method_dict:
            continue
        if method not in eb_plot_methods:
            continue
        ls = '--' if method in eb_baseline_methods else '-'
        ax.plot(best_curves_prog[method], error_bounds, linestyle=ls,
                color=method_colors[method], marker='.', label=method_dict[method],
                linewidth=1, markersize=4)
    ax.set_xlabel('Bitrate', fontsize=10)
    ax.set_ylabel('Error Bound', fontsize=10)
    ax.set_xlim(lim["xlim_eb"])
    ax.set_ylim(lim["ylim_eb"])
    ax.set_yscale('log')
    ax.set_title(f'{ds_name}', fontsize=11)

handles, labels = axes[0][0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=5, bbox_to_anchor=(0.5, -0.08), fontsize=9)
plt.tight_layout()
plt.subplots_adjust(bottom=0.1)
# plt.savefig('EB_vs_Bitrate_baseline.pdf', bbox_inches='tight')
plt.show()


In [ ]:
# Per-figure zoom limits.
AXES = {
    "CESM":    {"xlim_zoom": (1, 2),    "ylim_zoom": (3e-4, 7e-4)},
    "Miranda": {"xlim_zoom": (0.5, 1.7),"ylim_zoom": (3e-4, 7e-4)},
    "SCALE":   {"xlim_zoom": (0.9, 2),  "ylim_zoom": (3e-4, 7e-4)},
    "S3D":     {"xlim_zoom": (0.2, 0.9),"ylim_zoom": (3e-4, 7e-4)},
}

# Same ablation files as every figure; this plot uses indices 1/2/4/6 (6 = Ours/CP).
method_dict = {"1": "IPComp", "2": "PMGARD", "4": "SZ3-R", "6": "Ours"}
method_colors = {"1": "purple", "2": "brown", "4": "cyan", "6": "red"}

# EB baseline: 1=IPComp, 2=PMGARD-HB(eb), 4=SZ3-R, 7=CP
plot_methods = ["1", "2", "4", "6"]
baseline_methods = ["1", "2", "4"]
cp_method = "6"

fig, axes = plt.subplots(2, 2, figsize=(6, 4.5))

for idx, ds_name in enumerate(DATASET_ORDER):
    ax = axes[idx // 2][idx % 2]
    lim = AXES[ds_name]
    error_bounds, best_curves_prog, best_PSNR = load_dataset_filtered(ds_name)

    for method in sorted(best_curves_prog.keys()):
        if method not in method_dict:
            continue
        if method not in plot_methods:
            continue
        ls = '--' if method in baseline_methods else '-'
        ax.plot(best_curves_prog[method], error_bounds, linestyle=ls,
                color=method_colors[method], marker='.', label=method_dict[method],
                linewidth=1, markersize=4)
    ax.set_xlabel('Bitrate', fontsize=10)
    ax.set_ylabel('Error Bound', fontsize=10)
    ax.set_xlim(lim["xlim_zoom"])
    ax.set_ylim(lim["ylim_zoom"])
    ax.set_yscale('log')
    ax.set_title(f'{ds_name}', fontsize=11)

handles, labels = axes[0][0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=5, bbox_to_anchor=(0.5, -0.08), fontsize=9)

plt.tight_layout()
plt.subplots_adjust(bottom=0.1)
# plt.savefig('EB_vs_Bitrate_baseline_zoom_in.pdf', bbox_inches='tight')
plt.show()


### Fig. 10 — Baseline comparison, PSNR mode

Baseline comparison of PSNR mode.

In [ ]:
# Per-figure axis limits.
AXES = {
    "CESM":    {"xlim_psnr": (0, 8), "ylim_psnr": (40, 150)},
    "Miranda": {"xlim_psnr": (0, 8), "ylim_psnr": (60, 180)},
    "SCALE":   {"xlim_psnr": (0, 8), "ylim_psnr": (60, 160)},
    "S3D":     {"xlim_psnr": (0, 8), "ylim_psnr": (60, 180)},
}

# Same ablation files as every figure; this plot uses indices 1/3/4/8 (8 = Ours).
method_dict = {"1": "IPComp", "3": "PMGARD", "4": "SZ3-R", "8": "Ours"}
method_colors = {"1": "purple", "3": "brown", "4": "cyan", "8": "red"}

fig, axes = plt.subplots(2, 2, figsize=(6, 4.5))

for idx, ds_name in enumerate(DATASET_ORDER):
    ax = axes[idx // 2][idx % 2]
    lim = AXES[ds_name]
    error_bounds, best_curves_prog, best_PSNR = load_dataset_filtered(ds_name)

    for method in sorted(best_curves_prog.keys()):
        if method not in method_dict:
            continue
        if method in ["2", "5", "6", "7", "9"]:
            continue
        ls = '--' if method in ["1", "3", "4"] else '-'
        ax.plot(best_curves_prog[method], best_PSNR[method], linestyle=ls,
                color=method_colors[method], marker='.', label=method_dict[method],
                linewidth=1, markersize=4)
    ax.set_xlabel('Bitrate', fontsize=10)
    ax.set_ylabel('PSNR', fontsize=10)
    ax.set_xlim(lim["xlim_psnr"])
    ax.set_ylim(lim["ylim_psnr"])
    ax.set_title(f'{ds_name}', fontsize=11)

handles, labels = axes[0][0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=5, bbox_to_anchor=(0.5, -0.08), fontsize=9)

plt.tight_layout()
plt.subplots_adjust(bottom=0.1)
# plt.savefig('PSNR_vs_Bitrate_baseline.pdf', bbox_inches='tight')
plt.show()


In [ ]:
# Per-figure zoom limits.
AXES = {
    "CESM":    {"xlim_psnr": (0.05, 0.4), "ylim_psnr": (53, 60)},
    "Miranda": {"xlim_psnr": (0.1, 1),    "ylim_psnr": (75, 83)},
    "SCALE":   {"xlim_psnr": (0.05, 0.5), "ylim_psnr": (61, 68)},
    "S3D":     {"xlim_psnr": (0.5, 4.0),  "ylim_psnr": (115, 122)},
}

# Same ablation files as every figure; this plot uses indices 1/3/4/8 (8 = Ours/CP).
method_dict = {"1": "IPComp", "3": "PMGARD", "4": "SZ3-R", "8": "Ours"}
method_colors = {"1": "purple", "3": "brown", "4": "cyan", "8": "red"}

fig, axes = plt.subplots(2, 2, figsize=(6, 4.5))

for idx, ds_name in enumerate(DATASET_ORDER):
    ax = axes[idx // 2][idx % 2]
    lim = AXES[ds_name]
    error_bounds, best_curves_prog, best_PSNR = load_dataset_filtered(ds_name)

    for method in sorted(best_curves_prog.keys()):
        if method not in method_dict:
            continue
        if method in ["2", "5", "6", "7", "9"]:
            continue
        ls = '--' if method in ["1", "3", "4"] else '-'
        ax.plot(best_curves_prog[method], best_PSNR[method], linestyle=ls,
                color=method_colors[method], marker='.', label=method_dict[method],
                linewidth=1, markersize=4)
    ax.set_xlabel('Bitrate', fontsize=10)
    ax.set_ylabel('PSNR', fontsize=10)
    ax.set_xlim(lim["xlim_psnr"])
    ax.set_ylim(lim["ylim_psnr"])
    ax.set_title(f'{ds_name}', fontsize=11)

handles, labels = axes[0][0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=4, bbox_to_anchor=(0.5, -0.08), fontsize=9)

plt.tight_layout()
plt.subplots_adjust(bottom=0.1)
# plt.savefig('PSNR_vs_Bitrate_baseline_zoom_in.pdf', bbox_inches='tight')
plt.show()


### Table 3 — Average refactor time

Average refactor time (in seconds) of different progressive approaches using all fields from the same dataset.

In [ ]:
eb_methods = {"1": "IPComp", "2": "PMGARD", "4": "SZ3-R", "6": "Ours"}
psnr_methods = {"1": "IPComp", "3": "PMGARD", "4": "SZ3-R", "8": "Ours"}


def parse_refactor_times(result_dir, prefix, variables, suffix="_ablation.txt"):
    """Parse refactor time for each method across all variables, return {method_id: [(var, time)]}."""
    times = defaultdict(list)
    for var in variables:
        filename = f"{result_dir}/{prefix}_{var}{suffix}"
        try:
            for line in open(filename, 'r').readlines():
                line_s = line.strip()
                if "Compression time" in line_s:
                    method_id = line_s.split(",")[0].split("#")[1]
                    m = re.search(r'Compression time\s*=\s*([\d.]+)', line_s)
                    if m:
                        times[method_id].append((var, float(m.group(1))))
                    continue
                m = re.search(r'Refactor[_ ]time:\s*([\d.]+)', line_s)
                if m:
                    method_id = line_s.split(",")[0].split("#")[1]
                    times[method_id].append((var, float(m.group(1))))
        except FileNotFoundError:
            print(f"Warning: {filename} not found, skipping")
    return times


def avg_time(times_dict, mid):
    """Average refactor time (s) for one method id, or None if no data."""
    entries = times_dict.get(mid, [])
    return np.mean([t for _, t in entries]) if entries else None


# -----------------------------------------------------------------------------
# Collect per-dataset average refactor time for both modes.
#   eb_avg[name][dataset]   and   psnr_avg[name][dataset]
# SZ3-R and IPComp use the same method id in both modes, so their EB and PSNR
# values are identical -> shown once as "N/A" mode in the table.
# -----------------------------------------------------------------------------
eb_avg = defaultdict(dict)
psnr_avg = defaultdict(dict)

for ds_name in DATASET_ORDER:
    meta = DATASETS[ds_name]
    times = parse_refactor_times(result_dir(ds_name), meta["prefix"], meta["variables"],
                                 suffix="_ablation.txt")
    for mid, name in eb_methods.items():
        eb_avg[name][ds_name] = avg_time(times, mid)
    for mid, name in psnr_methods.items():
        psnr_avg[name][ds_name] = avg_time(times, mid)


# -----------------------------------------------------------------------------
# Print Table III
#   - "Our method" and "PMGARD" get EB and PSNR rows (mode-dependent ids)
#   - "SZ3-R" and "IPComp" get a single "N/A" row (same id in both modes)
# Row layout: (display method name, mode label, values dict)
# -----------------------------------------------------------------------------
rows = [
    ("Our method", "EB",   eb_avg["Ours"]),
    ("Our method", "PSNR", psnr_avg["Ours"]),
    ("PMGARD",     "EB",   eb_avg["PMGARD"]),
    ("PMGARD",     "PSNR", psnr_avg["PMGARD"]),
    ("SZ3-R",      "N/A",  eb_avg["SZ3-R"]),
    ("IPComp",     "N/A",  eb_avg["IPComp"]),
]

def fmt(v):
    return f"{v:.2f}" if v is not None else "-"

# Column widths
method_w = max(len("Method"), max(len(r[0]) for r in rows))
mode_w = max(len(m) for _, m, _ in rows)
ds_w = {ds: max(len(ds), 6) for ds in DATASET_ORDER}

print("TABLE III")
print("Average refactor time (in seconds) of different progressive")
print("approaches using all fields from the same dataset")
print()

# Header
header = f"{'Method':<{method_w}}  {'':<{mode_w}}  " + "  ".join(f"{ds:>{ds_w[ds]}}" for ds in DATASET_ORDER)
print(header)
print("-" * len(header))

# Body: blank the method name on the second row of a two-row method for readability
prev_method = None
for method, mode, vals in rows:
    label = method if method != prev_method else ""
    prev_method = method
    cells = "  ".join(f"{fmt(vals.get(ds)):>{ds_w[ds]}}" for ds in DATASET_ORDER)
    print(f"{label:<{method_w}}  {mode:<{mode_w}}  {cells}")

### Table 4 — Average reconstruction time and bit-rate

Average reconstruction time (in seconds) and average bit-rate (BR) of different progressive approaches using all fields from the same dataset under tolerance 10^-4.

In [ ]:
TARGET_EB = 0.0001

eb_methods = {"1": "IPComp", "2": "PMGARD", "4": "SZ3-R", "6": "Ours"}
psnr_methods = {"1": "IPComp", "3": "PMGARD", "4": "SZ3-R", "8": "Ours"}


def parse_reconstruct_data(result_dir, prefix, variables, target_eb, suffix="_ablation.txt"):
    """Parse reconstruct time and bitrate at a specific eb for each method.
    Returns {method_id: {'times': [float], 'bitrates': [float]}}."""
    data = defaultdict(lambda: {'times': [], 'bitrates': []})
    for var in variables:
        filename = f"{result_dir}/{prefix}_{var}{suffix}"
        try:
            for line in open(filename, 'r').readlines():
                line_s = line.strip()
                if "ErrorBound=" not in line_s:
                    continue
                eb = float(line_s.split("ErrorBound=")[1].split(",")[0])
                if abs(eb - target_eb) / target_eb > 0.01:
                    continue
                method_id = line_s.split(",")[0].split("#")[1]

                # Parse time
                time_val = None
                m1 = re.search(r'Decompression time\s*=\s*([\d.]+)', line_s)
                if m1:
                    time_val = float(m1.group(1))
                else:
                    m2 = re.search(r'Reconstruct[_ ]time:\s*([\d.]+)', line_s)
                    if m2:
                        time_val = float(m2.group(1))
                if time_val is not None:
                    data[method_id]['times'].append(time_val)

                # Parse bitrate
                mb = re.search(r'[Bb]it[_ ]?[Rr]ate[:\s=]*([\d.]+)', line_s)
                if mb:
                    data[method_id]['bitrates'].append(float(mb.group(1)))
                else:
                    # try "CR=" or "Compression ratio" and convert
                    mc = re.search(r'CR[:\s=]*([\d.]+)', line_s)
                    if mc:
                        cr = float(mc.group(1))
                        if cr > 0:
                            data[method_id]['bitrates'].append(64.0 / cr)
        except FileNotFoundError:
            print(f"Warning: {filename} not found, skipping")
    return data


def avg_time_br(data_dict, mid):
    """Return (avg_time, avg_bitrate) for one method id; None where no data."""
    d = data_dict.get(mid, {'times': [], 'bitrates': []})
    t = np.mean(d['times']) if d['times'] else None
    b = np.mean(d['bitrates']) if d['bitrates'] else None
    return t, b


# -----------------------------------------------------------------------------
# Collect per-dataset (time, bitrate) for both modes.
#   results[name][mode][dataset] = (time, br)
# SZ3-R and IPComp use the same method id in both modes -> single "N/A" row.
# -----------------------------------------------------------------------------
results = defaultdict(lambda: defaultdict(dict))

for ds_name in DATASET_ORDER:
    meta = DATASETS[ds_name]
    recon = parse_reconstruct_data(result_dir(ds_name), meta["prefix"], meta["variables"], TARGET_EB)
    for mid, name in eb_methods.items():
        results[name]["EB"][ds_name] = avg_time_br(recon, mid)
    for mid, name in psnr_methods.items():
        results[name]["PSNR"][ds_name] = avg_time_br(recon, mid)


# -----------------------------------------------------------------------------
# Print Table IV
#   - "Our method" and "PMGARD": EB and PSNR rows
#   - "SZ3-R" and "IPComp": single "N/A" row (same id in both modes)
# Each dataset spans two sub-columns: Time and BR.
# Row layout: (display name, mode label, {dataset: (time, br)})
# -----------------------------------------------------------------------------
rows = [
    ("Our method", "EB",   results["Ours"]["EB"]),
    ("Our method", "PSNR", results["Ours"]["PSNR"]),
    ("PMGARD",     "EB",   results["PMGARD"]["EB"]),
    ("PMGARD",     "PSNR", results["PMGARD"]["PSNR"]),
    ("SZ3-R",      "N/A",  results["SZ3-R"]["EB"]),
    ("IPComp",     "N/A",  results["IPComp"]["EB"]),
]

def fmt(v):
    return f"{v:.2f}" if v is not None else "-"

method_w = max(len("Method"), max(len(r[0]) for r in rows))
mode_w = max(len(m) for _, m, _ in rows)
col_w = 5  # width per Time / BR sub-column

print("TABLE IV")
print("Average reconstruction time (in seconds) and average bit-rate (BR) of")
print("different progressive approaches using all fields from the same dataset")
print(f"under tolerance tau = {TARGET_EB:g}")
print()

# Header row 1: dataset names spanning two sub-columns each
h1 = f"{'':<{method_w}}  {'':<{mode_w}}  " + "  ".join(
    f"{ds:^{col_w*2+2}}" for ds in DATASET_ORDER)
# Header row 2: Time / BR under each dataset
h2 = f"{'Method':<{method_w}}  {'':<{mode_w}}  " + "  ".join(
    f"{'Time':>{col_w}} {'BR':>{col_w}}" for _ in DATASET_ORDER)
print(h1)
print(h2)
print("-" * len(h2))

# Body
prev_method = None
for method, mode, vals in rows:
    label = method if method != prev_method else ""
    prev_method = method
    cells = "  ".join(
        f"{fmt(vals.get(ds, (None, None))[0]):>{col_w}} {fmt(vals.get(ds, (None, None))[1]):>{col_w}}"
        for ds in DATASET_ORDER)
    print(f"{label:<{method_w}}  {mode:<{col_w if mode_w<col_w else mode_w}}  {cells}"
          .replace(f"{mode:<{col_w if mode_w<col_w else mode_w}}", f"{mode:<{mode_w}}", 1))

### Fig. 12 — Visual comparison (CESM, Temperature field)

Fig. 12 compares the reconstructed Temperature field from the different
progressive approaches.

> **Prerequisite:** run this **after** the CESM ablation-study job (Fig. 7–10)
> has finished. IPComp and SZ3-R reuse the refactored representation produced
> there, so it must exist first. (PMGARD and Ours are re-refactored inside this
> job, since their reconstructed filenames would otherwise overwrite the
> ablation outputs.)

Generate the reconstructed data in one of two ways:

**Option 1 — submit it for you.** Run the job-submission cell below
(`submit_job("visualize_CESM_Temp.sh", ...)`); depending on your
`EXECUTION_MODE`, it either fills in your `SBATCH_SETTINGS` and submits via
sbatch, or queues the job to be run with GNU Parallel via `run_queued_jobs()`.

**Option 2 — do it yourself.**

*If `EXECUTION_MODE = "slurm"`:* edit the `#SBATCH` directives in
`./ae_scripts/visualize_CESM_Temp.sh`, then:

    cd ProDM
    sbatch ./ae_scripts/visualize_CESM_Temp.sh {DATA_ROOT}/CESM

*If `EXECUTION_MODE = "parallel"` (e.g. a single machine/bare-metal node
without SLURM):* no `#SBATCH` edits needed, just run it directly:

    cd ProDM
    bash ./ae_scripts/visualize_CESM_Temp.sh {DATA_ROOT}/CESM

Either way, the job reconstructs the `T` field at the per-method error bounds
used in the paper and writes the outputs to
`{DATA_ROOT}/CESM/refactor/T_refactored/`:

- `IPComp_T.dat`    — IPComp
- `PMGARD_T.dat`    — PMGARD
- `SZ3R_T_low.dat`  — SZ3-R (lower bit-rate)
- `SZ3R_T_high.dat` — SZ3-R (higher bit-rate)
- `TWO_T.dat`       — Ours

Once the job finishes, fill the paths in the Fig. 12 cell:

    ORI_PATH       = "{DATA_ROOT}/CESM/data/T.dat"
    IPCOMP_PATH    = "{DATA_ROOT}/CESM/refactor/T_refactored/IPComp_T.dat"
    PMGARD_PATH    = "{DATA_ROOT}/CESM/refactor/T_refactored/PMGARD_T.dat"
    SZ3R_PATH_LOW  = "{DATA_ROOT}/CESM/refactor/T_refactored/SZ3R_T_low.dat"
    SZ3R_PATH_HIGH = "{DATA_ROOT}/CESM/refactor/T_refactored/SZ3R_T_high.dat"
    OURS_PATH      = "{DATA_ROOT}/CESM/refactor/T_refactored/TWO_T.dat"

then run the cell to produce Fig. 12.

In [ ]:
# -----------------------------------------------------------------------------
# Step 3: submit the Fig. 12 visualization job (CESM Temperature).
#
# Reconstructs the T field from all methods at the paper's per-method error
# bounds, writing outputs to {DATA_ROOT}/CESM/refactor/T_refactored/.
# Requires the CESM ablation-study job (Fig. 7-10) to have finished first,
# since IPComp and SZ3-R reuse the refactored representation produced there.
# -----------------------------------------------------------------------------
cesm_path = os.path.join(DATA_ROOT, "CESM")
submit_job("visualize_CESM_Temp.sh", data_path=cesm_path)

if EXECUTION_MODE == "parallel":
    print("\n[parallel] Visualization job running...\
           \n Around 2 minutes on a single compute_gigaio node (CHI@UC, dual EPYC 7763)")
    run_queued_jobs()
    print("\n[parallel] Visualization job complete. "
          "On a single compute_gigaio node (CHI@UC, dual EPYC 7763) this takes about 2 minutes.")  
else:
    print("\n[SLURM] Visualization job submitted. "
          "Check status with: squeue -u $USER, and wait until they finish.\n"
          "runtime is approximately: ≈2 minutes.")

### Fig. 12 — Visualization

Visual comparison of reconstructed data from different progressive approaches on the CESM data, field Temperature, at the slice [11, 1000:1200, 1480:1680]. Each panel shows the method name, achieved bit-rate, global PSNR over the full field, and RoI PSNR computed within the displayed region.

In [ ]:
# =============================================================================
# Figure 12 - Visual comparison (CESM, Temperature field)
#
# Reconstructed fields and their bit-rate / PSNR are produced by
# ae_scripts/visualize_CESM_Temp.sh, which writes:
#   - reconstructed .dat files -> {DATA_ROOT}/CESM/refactor/T_refactored/
#   - a metrics log            -> ./ae_results/CESM_Results/CESM_T_visualization.txt
# This cell reads both automatically; no manual paths or numbers needed.
# =============================================================================

CESM_T_DIR = os.path.join(DATA_ROOT, "CESM", "refactor", "T_refactored")
VIS_LOG = "./ae_results/CESM_Results/CESM_T_visualization.txt"
SHAPE = (26, 1800, 3600)

def load_field(fname):
    return np.fromfile(os.path.join(CESM_T_DIR, fname), dtype=np.float64).reshape(SHAPE)

# Panels: (display name, .dat filename, Method#, ErrorBound tag used in the log)
# SZ3-R appears twice, distinguished by error bound (1e-2 = low, 1e-3 = high).
panel_specs = [
    ("Original", "T.dat_original", None, None),   # original handled specially below
    ("IPComp",   "IPComp_T.dat",   "1", "2e-3"),
    ("PMGARD",   "PMGARD_T.dat",   "3", "4e-3"),
    ("SZ3-R",    "SZ3R_T_low.dat", "4", "1e-2"),
    ("SZ3-R",    "SZ3R_T_high.dat","4", "1e-3"),
    ("Ours",     "TWO_T.dat",      "8", "2e-3"),
]

def _extract(line, key):
    """Pull the number following `key` (handles 'key = v', 'key= v', 'key: v', 'key v')."""
    m = re.search(rf"{key}\s*[:=]?\s*([-+]?\d*\.?\d+(?:[eE][-+]?\d+)?)", line)
    return float(m.group(1)) if m else None

def parse_metrics(log_path):
    """Return {(method, eb_tag): (bitrate, psnr)} from the visualization log."""
    metrics = {}
    for line in open(log_path):
        if "ErrorBound=" not in line or "Method#" not in line:
            continue
        method = line.split(",")[0].split("#")[1].strip()
        eb_tag = line.split("ErrorBound=")[1].split(",")[0].strip()
        br = _extract(line, "Bitrate")
        psnr = _extract(line, "PSNR")
        metrics[(method, eb_tag)] = (br, psnr)
    return metrics

metrics = parse_metrics(VIS_LOG)

# Load original once (the input data file), then each reconstruction.
ori_data = np.fromfile(os.path.join(DATA_ROOT, "CESM", "data", "T.dat"),
                       dtype=np.float64).reshape(SHAPE)

xi = 11
zy0, zy1 = 1000, 1200
zx0, zx1 = 1480, 1680

ori_patch = ori_data[xi, zy0:zy1, zx0:zx1]
vmin, vmax = ori_patch.min(), ori_patch.max()

# Assemble panels with BR/PSNR pulled from the log.
panels = []
for name, fname, method, eb_tag in panel_specs:
    if name == "Original":
        panels.append(("Original", None, None, None))
        continue
    dec = load_field(fname)
    br, gpsnr = metrics.get((method, eb_tag), (None, None))
    panels.append((name, dec, br, gpsnr))

# Layout: 2 rows x 3 cols
fig, axes = plt.subplots(2, 3, figsize=(5.2, 4.2))

# ---- title: three lines -> method (+BR) / PSNR / RoI PSNR ----
for ax, (name, dec, br, gpsnr) in zip(axes.flat, panels):
    if name == "Original":
        patch = ori_patch
        title = "Original"
    else:
        patch = dec[xi, zy0:zy1, zx0:zx1]
        mse = np.mean((ori_patch - patch) ** 2)
        dr = vmax - vmin
        roi_psnr = 10 * np.log10(dr**2 / mse) if mse > 0 else float('inf')
        br_str = f"{br:.2f}" if br is not None else "?"
        psnr_str = f"{gpsnr:.1f}" if gpsnr is not None else "?"
        # three lines
        title = f"{name} (BR={br_str})\nPSNR = {psnr_str} dB\nRoI PSNR = {roi_psnr:.1f} dB"

    ax.imshow(patch, aspect='equal', cmap='coolwarm', vmin=vmin, vmax=vmax)
    ax.set_title(title, fontsize=5.5, pad=2, linespacing=1.25)
    ax.set_xticks([])
    ax.set_yticks([])

fig.subplots_adjust(left=0.02, right=0.88, wspace=0.15, hspace=0.55, top=0.88, bottom=0.02)
cbar_ax = fig.add_axes([0.90, 0.10, 0.03, 0.75])
im = axes[0, 0].images[0]
fig.colorbar(im, cax=cbar_ax, label='T')
cbar_ax.yaxis.label.set_size(7)
cbar_ax.tick_params(labelsize=6)
# plt.savefig('CESM_T_comparison_column.pdf', dpi=300, bbox_inches='tight')
# plt.savefig('CESM_T_comparison_column.png', dpi=300, bbox_inches='tight')
plt.show()

## Part 2 - Optional: JHTDB Weak-Scaling Study (Fig.11)

### Fig. 11 — Dataset preparation (JHTDB) **Optional**

**Note**

> ⚠️ **About JHTDB access (please read first).**
> We observed a period during which the JHTDB `rotstrat4096` dataset was **not
> reliably reachable through JHTDB's own programmatic interface** — its legacy
> SOAP interface had been retired, and the new cutout service (the
> `giverny`/`givernylocal` REST endpoint) was returning server-side errors
> (HTTP 500) for this dataset (verified with multiple tokens, datasets, and
> request sizes) at the time. This appeared to be an issue on the JHTDB side,
> not with our artifact, and may since have been resolved.
>
> **If you run into this unavailability, we provide the exact original data
> used in our experiments via Globus as a fallback.** You do not need a JHTDB
> token or the download block below in that case — simply transfer our shared
> collection directly to your cluster:
>
> **Globus shared link:** https://app.globus.org/file-manager?origin_id=601f294f-d6a8-44b6-b9df-a25b540d0ab3&origin_path=%2F valid within 60 days since Jul 8, 2026 due to cluster storage policy.
>
> This collection contains the 1024 cubes (256×512×512) of the rotstrat4096
> temperature field, which serve as the original data for the parallel-refactoring
> pipeline.
>
> **Licensing / attribution.** JHTDB data is released under the Open Data Commons
> Attribution License (ODC-By, https://opendatacommons.org/licenses/by/), which
> permits free use, modification, and redistribution (including commercially),
> provided the source is attributed. Redistributing this subset is therefore
> permitted. For your convenience, we include a `license.txt` in the Globus
> collection that records the ODC-By license and the suggested citation(s) for the
> `rotstrat4096` dataset; please retain it and cite the referenced work if you use
> the data. See https://turbulence.pha.jhu.edu/citing.aspx for JHTDB's citation
> guidance. JHU and the parties involved in producing the data assume no liability
> for its use; use is at your own risk.
>
> To transfer: log in to Globus, set up an endpoint (or Globus Connect Personal)
> at your storage location, open the link above, and transfer the collection to
> your `DATA_ROOT`. From these cubes, the remaining derived data (per-core-count
> copies, refactored representations, and retrieved data) are generated on your
> cluster by the scripts provided below.

We recommend first trying to obtain the data directly from JHTDB at
https://turbulence.idies.jhu.edu/ (dataset: `rotstrat4096`, temperature field),
which requires a JHTDB authorization [token](https://turbulence.idies.jhu.edu/database).
The download block below is provided for that case. If you encounter the
unavailability described above, fall back to the Globus data instead.

The cubes themselves are 512 GB, but reproducing Fig. 11 needs substantially more
storage — about **2.3 TB** in total, because several derived forms of the data
must coexist:

- **Cubed original data for parallel refactoring (~512 GB)** — the 1024 cubes of
  256×512×512 provided via Globus above.
- **Refactored representations (~1536 GB)** — output of **all four progressive approaches**.
- **Retrieved data (~276 GB)** — retrieved by the four approaches, kept for transfer.

Please ensure at least ~2.3 TB of free space before running the JHTDB experiment.

> ⚠️ **About compute resources (please read before submitting).**
> Reproducing Fig. 11 is compute-intensive: the weak-scaling study runs all four
> methods across `CORE_COUNTS = [128, 256, 512, 1024]`, so the largest configuration
> alone requires **1024 CPU cores** available concurrently (25 nodes at 64 cores/node
> on our cluster).
>
> We ran these experiments on a cluster with **100+ compute nodes**, each configured
> identically with **two AMD EPYC ROME 7702P processors (64 cores each) and 256 GB of
> memory** per node. This homogeneity — every node identical in CPU model, core count,
> and memory — is what let us obtain clean, comparable weak-scaling numbers across
> core counts.
>
> **Many clusters do not have this kind of uniformity** (mixed node generations,
> varying core counts per node, heterogeneous memory, or shared/oversubscribed
> nodes). If your cluster is heterogeneous, the specific runtime numbers you get
> may not match ours exactly, and comparisons across core counts may be less
> apples-to-apples. This does not affect correctness of the pipeline, but please
> keep this in mind when interpreting absolute timing results.
>
> **Queueing time.** Because the 1024-core configuration requests a large,
> contiguous allocation on a shared cluster, it may **not start immediately** —
> depending on current cluster load, the scheduler may queue this job for
> several hours to over a day before enough nodes become free
> (`squeue -u $USER` will show `Reason=Resources` and an estimated `StartTime`
> while it waits). This is expected behavior, not a failure. Please plan
> evaluation time accordingly, and consider submitting this job early / in
> the background while working through other figures. If your scheduler
> supports it, requesting a homogeneous node allocation (e.g., via
> `--constraint`/`--exclusive` in Slurm) can also help both scheduling
> predictability and result comparability.

In [ ]:
# ---- REVIEWER: here tells where to store the original data of JHTDB --------------------------------
print("Transfer from Globus shared link to:", os.path.join(DATA_ROOT, "rotstrat4096_temp_d64_1024/data"))
# If path does not exist, create first.
os.makedirs(os.path.join(DATA_ROOT, "rotstrat4096_temp_d64_1024/data"), exist_ok=True)
# -----------------------------------------------------------------------------

> The 2 cells below are the script we originally used to download the cubes from
> JHTDB, included here for reference. If JHTDB's programmatic interface is
> unavailable when you try this (see the note above), the cells will not run —
> in that case, **please use the Globus data instead**; no token or download
> needed.

In [ ]:
# ---- REVIEWER: put your token here --------------------------------
TOKEN = "" # Replace with your actual token
# -----------------------------------------------------------------------------

In [ ]:
# =============================================================================
# Fig. 11 — JHTDB download block (FALLBACK / reference only).
#
# Downloads rotstrat4096 temperature as 1024 cubes of 256x512x512, k-fastest,
# named Temperature_0 .. Temperature_1023, converted to float64 (.d64).
#
# NOTE: This uses JHTDB's programmatic interface, which is currently returning
# server-side errors for this dataset (see the availability note above). Prefer
# the Globus shared data. This block is provided for when JHTDB access is restored.
# Requires a JHTDB token.
# =============================================================================
import os
import base64
import requests
import xml.etree.ElementTree as ET
from multiprocessing import Pool, Manager
import time
import random

# ---- config -----------------------------------------------------------------
DATASET = "rotstrat4096"
FIELD = "temperature"
SNAPSHOT = 5                    # rotstrat4096 snapshot number
FILTER_WIDTH = 1
ADDR = "none"

# Original volume 4096^3, split into cubes of 256 x 512 x 512.
BLOCK_X, BLOCK_Y, BLOCK_Z = 256, 512, 512
NUM_X = 4096 // BLOCK_X         # 16
NUM_Y = 4096 // BLOCK_Y         # 8
NUM_Z = 4096 // BLOCK_Z         # 8
TOTAL_CUBES = NUM_X * NUM_Y * NUM_Z   # 1024

X_STEP = Y_STEP = Z_STEP = T_STEP = 1

# DATA_ROOT is defined in the data-preparation cell above.
OUTDIR = os.path.join(DATA_ROOT, "rotstrat4096_temp_d64_1024", "data")
PREFIX = "Temperature"
URL = "https://turbulence.pha.jhu.edu/service/turbulence.asmx"
HEADERS = {"Content-Type": "application/soap+xml; charset=utf-8"}
LOG_FILE = os.path.join(DATA_ROOT, "rotstrat4096_temp_d64_1024", "downloaded.txt")
# -----------------------------------------------------------------------------

os.makedirs(OUTDIR, exist_ok=True)


def build_soap_body(x_start, x_end, y_start, y_end, z_start, z_end, t_start, t_end):
    return f"""<?xml version="1.0" encoding="utf-8"?>
<soap12:Envelope xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance"
                 xmlns:xsd="http://www.w3.org/2001/XMLSchema"
                 xmlns:soap12="http://www.w3.org/2003/05/soap-envelope">
  <soap12:Body>
    <GetAnyCutoutWeb xmlns="http://turbulence.pha.jhu.edu/">
      <authToken>{TOKEN}</authToken>
      <dataset>{DATASET}</dataset>
      <field>{FIELD}</field>
      <T>{SNAPSHOT}</T>
      <x_start>{x_start}</x_start>
      <y_start>{y_start}</y_start>
      <z_start>{z_start}</z_start>
      <t_start>{t_start}</t_start>
      <x_end>{x_end}</x_end>
      <y_end>{y_end}</y_end>
      <z_end>{z_end}</z_end>
      <t_end>{t_end}</t_end>
      <x_step>{X_STEP}</x_step>
      <y_step>{Y_STEP}</y_step>
      <z_step>{Z_STEP}</z_step>
      <t_step>{T_STEP}</t_step>
      <filter_width>{FILTER_WIDTH}</filter_width>
      <addr>{ADDR}</addr>
    </GetAnyCutoutWeb>
  </soap12:Body>
</soap12:Envelope>"""


def download_block(args):
    (n, i, j, k), downloaded_set, lock = args
    tag = f"{PREFIX}_{n}"                       # Temperature_0 .. Temperature_1023
    filepath = os.path.join(OUTDIR, tag + ".d64")

    if tag in downloaded_set or os.path.exists(filepath):
        print(f"[SKIP] {tag}")
        if tag not in downloaded_set:
            with lock:
                with open(LOG_FILE, "a") as log:
                    log.write(tag + "\n")
        return

    x_start = i * BLOCK_X + 1
    y_start = j * BLOCK_Y + 1
    z_start = k * BLOCK_Z + 1
    x_end = x_start + BLOCK_X - 1
    y_end = y_start + BLOCK_Y - 1
    z_end = z_start + BLOCK_Z - 1
    t_start = SNAPSHOT
    t_end = SNAPSHOT

    soap_body = build_soap_body(x_start, x_end, y_start, y_end, z_start, z_end, t_start, t_end)

    max_retries = 5
    delay = 2
    for attempt in range(max_retries):
        try:
            with requests.Session() as session:
                response = session.post(URL, data=soap_body, headers=HEADERS)
            if response.status_code == 200:
                tree = ET.fromstring(response.content)
                ns = {'soap': 'http://www.w3.org/2003/05/soap-envelope',
                      'jhtdb': 'http://turbulence.pha.jhu.edu/'}
                result = tree.find('.//jhtdb:GetAnyCutoutWebResult', ns)
                if result is not None and result.text:
                    # source is float32; convert to float64 (.d64)
                    raw = base64.b64decode(result.text)
                    arr = np.frombuffer(raw, dtype=np.float32).astype(np.float64)
                    arr.tofile(filepath)
                    print(f"[DONE] {tag}  ({arr.size} values)")
                    with lock:
                        with open(LOG_FILE, "a") as log:
                            log.write(tag + "\n")
                    return
                else:
                    raise Exception("No data in SOAP response")
            else:
                raise Exception(f"HTTP {response.status_code}")
        except Exception as e:
            print(f"[RETRY {attempt+1}] {tag} failed: {e}")
            sleep_time = delay + random.uniform(0, 1)
            print(f"Sleeping {sleep_time:.1f}s before retrying...")
            time.sleep(sleep_time)
            delay = min(delay * 2, 64)

    print(f"[FAIL] {tag} permanently failed after {max_retries} retries.")


# Build job list with linear index n, k fastest: n = i*(NUM_Y*NUM_Z) + j*NUM_Z + k
job_list = []
n = 0
for i in range(NUM_X):
    for j in range(NUM_Y):
        for k in range(NUM_Z):
            job_list.append((n, i, j, k))
            n += 1
# job_list = job_list[:1]   # quick test: one cube

# Load already-downloaded tags.
if os.path.exists(LOG_FILE):
    with open(LOG_FILE) as f:
        downloaded = set(line.strip() for line in f)
else:
    downloaded = set()

if not TOKEN:
    raise SystemExit("Set TOKEN to your JHTDB authorization token before running.")

with Manager() as manager:
    lock = manager.Lock()
    downloaded_set = manager.dict({tag: True for tag in downloaded})
    tasks = [(job, downloaded_set, lock) for job in job_list]
    with Pool(processes=16) as pool:
        pool.map(download_block, tasks)

print(f"\nDone. Cubes written to {OUTDIR}")

### Job Submission — JHTDB weak-scaling study (Fig. 11)

Then submit the JHTDB weak-scaling job, which demonstrates the **eb mode**
use case for end-to-end data transfer.

    sbatchh ./ae_scripts/submit_JHTDB.sh {JHTDB_ROOT}

This loops over `METHODS = [PMGARD, SZ3-R, IPComp, TWO]` and
`CORE_COUNTS = [128, 256, 512, 1024]`, refactoring each of the 1024 cubes
and reconstructing them at each core count. `submit_JHTDB.sh` carries the
`#SBATCH` directives, so that's the script whose header you need to fill in
(same fields as before).

Alternatively, use the two job-submission cells below, which fills the SBATCH
headers from your `SBATCH_SETTINGS` and submits the job for you.

Refactored (intermediate) representations for each of the 1024 cubes are
written to `{JHTDB_ROOT}/{0..1023}/`, and retrieved data for each
(method, core count) combination are written to
`{JHTDB_ROOT}/{method}/{cores}/`.

Wait until the job finishes (`squeue -u $USER`) before running the Fig. 11 cell.

In [ ]:
# =============================================================================
# Fig. 11 (JHTDB weak scaling) — cluster settings SPECIFIC to this job.
#
# Fig. 11 is a large MPI weak-scaling run (up to 1024 cores across nodes), so it
# needs its own settings, separate from the single-node SBATCH_SETTINGS defined
# earlier. Fill these in for a cluster/partition that can actually provide 1024
# cores of the SAME architecture (a homogeneous partition). Small heterogeneous
# partitions typically CANNOT run this figure.
# -----------------------------------------------------------------------------

# MPI module setup: submit_JHTDB.sh loads an MPI module before mpirun. The name
# is cluster-specific. Find it with `module avail mpi` or `module spider openmpi`.
# Examples:
#   MODULE_SETUP = "module purge\nmodule load openmpi/4.1.5"
#   MODULE_SETUP = "module purge\nmodule load mpi/hpcx"
#   MODULE_SETUP = "module load gcc/12 openmpi/5.0.0"
# Leave as "" only if mpirun is already on PATH without loading anything.
MODULE_SETUP = "module purge\nmodule load mpi/latest"   # <- edit for your cluster

# SBATCH settings for the JHTDB job. These OVERRIDE / supplement the template's
# #SBATCH header (submit_JHTDB.sh uses --ntasks=1024). Same fill/append rules as
# SBATCH_SETTINGS: options in the template are replaced in place, others are
# appended; use " " for valueless flags; leave "" to skip.
JHTDB_SBATCH_SETTINGS = {
    # --- required ---
    "--partition":  "",          # a partition that can supply 1024 cores
    "-A":           "",          # your allocation account

    # --- notifications ---
    "--mail-user":  "",          # your email (leave "" to drop the mail lines)

    # --- scale (must total 1024 MPI tasks; adjust to your node core counts) ---
    "--ntasks":         "1024",  # total MPI ranks for the weak-scaling run
    # "--nodes":        "",      # e.g. "16" if each node has 64 usable cores
    # "--ntasks-per-node": "",   # e.g. "64"
    # "--time":         "03:00:00",

    # --- node architecture / placement ---
    # Pin to the SAME architecture you compiled on (homogeneous nodes).
    "--constraint": "",          # e.g. "epyc7763", "cascadelake"

    # --- optional: uncomment as needed ---
    # "--exclusive":  " ",
    # "--mem-per-cpu": "12G",
    # "--nodelist":   "",
    # "--exclude":    "",
    # "--qos":        "",
}

In [ ]:
# -----------------------------------------------------------------------------
# Fig. 11 — submit the JHTDB weak-scaling job.
#
# Requires the 1024 cubes at {JHTDB_ROOT}/data/Temperature_0.d64 .. _1023.d64
# (from Globus, or the fallback download block above).
#
# The job writes per-method, per-core-count outputs to
#   {JHTDB_ROOT}/{method}/{cores}/
# so those directories are created here first (mpirun will not create them).
# -----------------------------------------------------------------------------
JHTDB_ROOT = os.path.join(DATA_ROOT, "rotstrat4096_temp_d64_1024")

# Create 0~1023 directories to hold the refactored representation of each cube.
for i in range(1024):
    os.makedirs(os.path.join(JHTDB_ROOT, str(i)), exist_ok=True)

print(f"Created {1024} refactored-representation directories under {JHTDB_ROOT}")

METHODS = ["PMGARD", "SZ3-R", "IPComp", "TWO"]
CORE_COUNTS = [128, 256, 512, 1024]

# Create output directories for every (method, core count).
for method in METHODS:
    for np_cores in CORE_COUNTS:
        os.makedirs(os.path.join(JHTDB_ROOT, method, str(np_cores)), exist_ok=True)

# Also ensure the results directory the script writes its log into exists.
os.makedirs("./ae_results/JHTDB_results", exist_ok=True)

print(f"Created {len(METHODS) * len(CORE_COUNTS)} output directories under {JHTDB_ROOT}")

# Submit the weak-scaling job; $1 = the cube directory.
submit_job("submit_JHTDB.sh", data_path=JHTDB_ROOT, module_setup=MODULE_SETUP, sbatch_settings=JHTDB_SBATCH_SETTINGS)

print("\nSubmitted JHTDB weak-scaling job. Check status with:  squeue -u $USER")
print(f"\nOnce finished, reconstructed files for each (method, core count) will be at:")
print(f"  {JHTDB_ROOT}/{{method}}/{{cores}}/")
print(f"  e.g. {os.path.join(JHTDB_ROOT, METHODS[0], str(CORE_COUNTS[0]))}")
print(f"Refactored (intermediate) representations for each cube are under:")
print(f"  {JHTDB_ROOT}/{{0..1023}}/")

### Fig. 11 — End-to-end data transfer (JHTDB)

End-to-end data transfer time using the JHTDB dataset. Transferring the original data of 128, 256, 512, and 1024 cores takes 189, 373, 735, and 1342 seconds, respectively.

In [ ]:
# =============================================================================
# Figure 11 - End-to-end data transfer time (JHTDB, weak scaling)
#
# retrieveTime is read from the parallel weak-scaling result file.
# retrieved size is also read from the same file (used by transfer Option 2).
#
# transferTime -- AE reviewers have TWO options:
#   Option 1 (measured):  actually transfer the retrieved data over Globus and
#                         record the wall-clock time; fill TRANSFER_MEASURED.
#   Option 2 (idealized): estimate transfer time as
#                             retrieved_size_bytes / TRANSFER_SPEED_BYTES_PER_SEC
#                         assuming an ideal constant transfer speed.
# Set USE_MEASURED_TRANSFER below to choose.
# =============================================================================

# -----------------------------------------------------------------------------
# Transfer time
# -----------------------------------------------------------------------------
USE_MEASURED_TRANSFER = False   # <- set True to use Option 1 (measured over Globus)

PARALLEL_RESULT_FILE = f"{AE_ROOT}/JHTDB_results/results_weak_scale.txt"

core_counts = np.array([128, 256, 512, 1024])

# Method order in the parallel log:
#   Method1 = PMGARD-HB, Method2 = PSZ3delta, Method3 = IPComp, Method4 = TWO (Ours)
METHOD_ID_TO_KEY = {"1": "pmgard", "2": "psz3", "3": "ipcomp", "4": "two"}
CORE_ORDER = [128, 256, 512, 1024]


def parse_weak_scale(filepath):
    """Parse the weak-scaling result file.

    Returns two dicts keyed by method ('pmgard','psz3','ipcomp','two'):
        retrieve[method][cores] = retrieval time (s)
        size[method][cores]     = retrieved size (bytes)

    The parser is tolerant of small format differences: it looks for a
    method id (e.g. 'Method#1' or 'Method1'), a core/process count, a
    retrieval time, and a retrieved size on each line. Adjust the regexes
    below if your file uses different field names.
    """
    retrieve = {v: {} for v in METHOD_ID_TO_KEY.values()}
    size = {v: {} for v in METHOD_ID_TO_KEY.values()}
    with open(filepath, "r") as f:
        for line in f:
            s = line.strip()
            if not s:
                continue
            m_id = re.search(r"Method#?(\d+)", s)
            if not m_id or m_id.group(1) not in METHOD_ID_TO_KEY:
                continue
            key = METHOD_ID_TO_KEY[m_id.group(1)]

            m_core = re.search(r"(?:Cores?|Procs?|nprocs|Processes?)\s*[=:]\s*(\d+)", s, re.I)
            m_time = re.search(r"(?:Retrieve|Retrieval|Read)[_ ]?time\s*[=:]\s*([\d.]+)", s, re.I)
            m_size = re.search(r"(?:Retrieved[_ ]?size|Size)\s*[=:]\s*([\d.]+)", s, re.I)

            if m_core is None or m_time is None:
                continue
            cores = int(m_core.group(1))
            retrieve[key][cores] = float(m_time.group(1))
            if m_size is not None:
                size[key][cores] = float(m_size.group(1))
    return retrieve, size


retrieve_raw, size_raw = parse_weak_scale(PARALLEL_RESULT_FILE)


def as_array(d, key):
    """Extract a value array in CORE_ORDER; NaN where missing."""
    return np.array([d[key].get(c, np.nan) for c in CORE_ORDER], dtype=float)


retrieveTime_pmgard = as_array(retrieve_raw, "pmgard")
retrieveTime_psz3   = as_array(retrieve_raw, "psz3")
retrieveTime_ipcomp = as_array(retrieve_raw, "ipcomp")
retrieveTime_two    = as_array(retrieve_raw, "two")

retrievedSize_pmgard = as_array(size_raw, "pmgard")
retrievedSize_psz3   = as_array(size_raw, "psz3")
retrievedSize_ipcomp = as_array(size_raw, "ipcomp")
retrievedSize_two    = as_array(size_raw, "two")

# Idealized transfer speed for Option 2 (bytes/sec). Change if desired.
TRANSFER_SPEED_BYTES_PER_SEC = 460239797

# --- Option 1: measured transfer times (fill in after running Globus) ---------
# One value per core count, in CORE_ORDER = [128, 256, 512, 1024].
TRANSFER_MEASURED = {
    "pmgard": np.array([np.nan, np.nan, np.nan, np.nan]),
    "psz3":   np.array([np.nan, np.nan, np.nan, np.nan]),
    "ipcomp": np.array([np.nan, np.nan, np.nan, np.nan]),
    "two":    np.array([np.nan, np.nan, np.nan, np.nan]),
}

# --- Option 2: idealized transfer time = retrieved_size / speed ---------------
def ideal_transfer(size_arr):
    return size_arr / TRANSFER_SPEED_BYTES_PER_SEC

if USE_MEASURED_TRANSFER:
    transferTime_pmgard = TRANSFER_MEASURED["pmgard"]
    transferTime_psz3   = TRANSFER_MEASURED["psz3"]
    transferTime_ipcomp = TRANSFER_MEASURED["ipcomp"]
    transferTime_two    = TRANSFER_MEASURED["two"]
else:
    transferTime_pmgard = ideal_transfer(retrievedSize_pmgard)
    transferTime_psz3   = ideal_transfer(retrievedSize_psz3)
    transferTime_ipcomp = ideal_transfer(retrievedSize_ipcomp)
    transferTime_two    = ideal_transfer(retrievedSize_two)

# -----------------------------------------------------------------------------
# Plot (stacked: retrieval + transfer), grouped by core count.
# -----------------------------------------------------------------------------
# Stack order: [TWO, PMGARD-HB, PSZ3delta, IPComp]
segment1 = np.column_stack((retrieveTime_two, retrieveTime_pmgard, retrieveTime_psz3, retrieveTime_ipcomp))
segment2 = np.column_stack((transferTime_two, transferTime_pmgard, transferTime_psz3, transferTime_ipcomp))

n_groups = 4
n_bars_per_group = 4
bar_width = 0.16
space_between_groups = 0.3
space_between_bars = 0.02

colors = ['r', '#eebe42', '#4998C7', 'g']
tags = [
    'Ours Retrieval', 'Ours Transfer',
    'PMGARD Retrieval', 'PMGARD Transfer',
    'SZ3-R Retrieval', 'SZ3-R Transfer',
    'IPComp Retrieval', 'IPComp Transfer'
]

fig, ax = plt.subplots(figsize=(6, 4.25))
index = np.arange(n_groups) * (n_bars_per_group * bar_width + space_between_groups)

for i in range(n_bars_per_group):
    bar_positions = index + i * (bar_width + space_between_bars)
    ax.bar(bar_positions, segment1[:, i], bar_width, color=colors[i], label=tags[i*2])
    ax.bar(bar_positions, segment2[:, i], bar_width, bottom=segment1[:, i], alpha=0.5, color=colors[i], label=tags[i*2+1])

ymax = np.nanmax(segment1 + segment2) * 1.05
ax.set_ylim(0, ymax)
ax.set_xlabel('Number of Cores (Data Size)', fontsize=12)
ax.set_ylabel('Total Time (s)', fontsize=12)
ax.set_title('End-to-End Data Transfer Time', fontsize=12)
ax.set_xticks(index + (n_bars_per_group * bar_width + 2 * space_between_bars) / 4)
ax.set_xticklabels(['128 (64GB)', '256 (128GB)', '512 (256GB)', '1024 (512GB)'], fontsize=10)
ax.tick_params(axis='y', labelsize=12)
ax.tick_params(axis='x', which='both', length=0)

ax.legend(prop={'size': 8.5}, loc='upper left', frameon=True, ncol=2)

plt.tight_layout()
# plt.savefig("weak_scale_JHTDB_1024.pdf", bbox_inches="tight")
plt.show()

# -----------------------------------------------------------------------------
# Print summary
# -----------------------------------------------------------------------------
print("=== Retrieve times (s) [128, 256, 512, 1024] ===")
print(f"PMGARD-HB:  {retrieveTime_pmgard}")
print(f"PSZ3delta:  {retrieveTime_psz3}")
print(f"IPComp:     {retrieveTime_ipcomp}")
print(f"TWO:        {retrieveTime_two}")

mode = "measured (Globus)" if USE_MEASURED_TRANSFER else \
       f"idealized ({TRANSFER_SPEED_BYTES_PER_SEC} B/s)"
print(f"\n=== Transfer times (s) -- {mode} ===")
print(f"PMGARD-HB:  {transferTime_pmgard}")
print(f"PSZ3delta:  {transferTime_psz3}")
print(f"IPComp:     {transferTime_ipcomp}")
print(f"TWO:        {transferTime_two}")

print("\n=== Total Times (retrieve + transfer) ===")
total_pmgard = retrieveTime_pmgard + transferTime_pmgard
total_psz3   = retrieveTime_psz3 + transferTime_psz3
total_ipcomp = retrieveTime_ipcomp + transferTime_ipcomp
total_two    = retrieveTime_two + transferTime_two
print(f"PMGARD-HB:  {total_pmgard}")
print(f"PSZ3delta:  {total_psz3}")
print(f"IPComp:     {total_ipcomp}")
print(f"TWO:        {total_two}")

print("\n=== Speedup (Method / TWO) ===")
print(f"PMGARD-HB / TWO:  {total_pmgard / total_two}")
print(f"PSZ3delta / TWO:  {total_psz3 / total_two}")
print(f"IPComp / TWO:     {total_ipcomp / total_two}")
